# ThyroFuse — Round-2 revision: every required run, in one notebook

This notebook produces all the verified evidence for the revised manuscript and the point-by-point response. Every analysis runs from **your own audited code and frozen manifest** (`revision_final_frozen_70_15_15`, SHA-256 `83e656ef…`). Nothing earlier is overwritten.

| Stage | What it produces | Reviewer comments | Compute |
|---|---|---|---|
| A | Cohort flow, duplicate and label-conflict tables, pHash sensitivity, measured leakage in the original split, reconciliation with the published 384-patient paper | R1-2, R2-5, R3-2, R5, R6-1, R6-3 | CPU, minutes |
| F | Frozen test set: all 5 backbones + ensemble, image and case level, validation thresholds, case-clustered 95% CIs, calibration and temperature scaling | R2-2, R2-6, R3-4, R5, R6-5 | CPU, minutes |
| H | External validation re-done correctly: official TN3K labels; original DDTI with its TI-RADS field; pre-specified label direction; cross-dataset duplicate screen | R1-1, R2-1, R2-7, R3-1, R3-5, R6-6 | GPU, ~15 min |
| G | Grad-CAM with quantified attribution outside the ultrasound sector; clean figure | R5, R6-10, R7-2 | GPU, ~5 min |
| I | Feature-distribution analysis across cohorts (t-SNE, MMD with permutation test) | R1-3 | GPU, ~5 min |
| B | Grouped 5-fold CV for **all** backbones, both modalities (your script 03b) | R2-5, R2-8, R5, R6-5, R7-6 | GPU, ~12–16 h |
| C | CV ensemble with thresholds from inner validation only; fold-level tables | R2-4, R2-5 | GPU, ~20 min |
| D | Leakage demonstration: identical code, image-level vs case-grouped folds (cytology) | R1-2, R4, R5 | GPU, ~4–5 h |
| E | Shortcut control: sector masked vs surround masked (ultrasound) | R5 | GPU, ~2–3 h |
| J–K | Publication figures, release package for GitHub/Zenodo, and **one results zip to upload** | R3-3, R5, R6-7, R6-10 | CPU, minutes |

## Before you run
1. **Download two public datasets** (both are needed for Stage H; everything else runs without them).
   - **TN3K classification labels.** From the TN3K authors' "Complete Dataset" Google Drive (linked on `huggingface.co/datasets/haifan-gong/TN3K` and the TRFE-Net GitHub), copy `label4test.csv` (and `label4trainval.csv` if present) into `D:\Thyroid-BHI-26\Thyroid Dataset\tn3k\`. That folder already holds `test-image\` and `trainval-image\`.
   - **Original DDTI with XML.** Download `dasmehdixtr/ddti-thyroid-ultrasound-images` from Kaggle and extract it to `D:\Thyroid-BHI-26\Thyroid Dataset\DDTI_original\`. Do **not** use `DDTI dataset\1_or_data`, whose own readme says its `category.csv` is not valid for classification.
2. Check `TORCH_PY` in CONFIG: it must be the Python that trained your models, i.e. `C:\Users\admin\miniconda3\envs\nnunet\python.exe`.
3. **Kernel → Restart & Run All.** Every stage is resumable. If the machine sleeps or you stop it, run again and finished work is skipped. Keep the PC awake overnight for Stages B–E.
4. At the end, upload **`D:\Thyroid-BHI-26\revision_round2\REVISION_ROUND2_UPLOAD.zip`**.

Everything printed is also written to `revision_round2\logs\`. Every table goes to `revision_round2\RESULTS.md`.

## CONFIG

In [ ]:
from pathlib import Path

ROOT            = Path(r"D:\Thyroid-BHI-26")
TORCH_PY        = Path(r"C:\Users\admin\miniconda3\envs\nnunet\python.exe")   # env that trained the models

FROZEN_MANIFEST = ROOT / "revision_final_frozen_70_15_15" / "04_frozen_manifest.csv"
FROZEN_SHA256   = "83e656eff33e272020d85f8faf4004285a67768d3ef79e040c59ee13ea1575f5"
EXCLUDED_CSV    = ROOT / "revision_final_frozen_70_15_15" / "01_excluded_cross_label_conflicts.csv"
SCRIPT_03B      = ROOT / "Rivision scripts" / "03b_group_stratified_5fold_cv.py"
SCRIPT_02       = ROOT / "Rivision scripts" / "02_experiment1b_retrain_five_backbones.py"
EXP1B_DIR       = ROOT / "revision_exp1b"
CV_DIR          = ROOT / "revision_exp2b_group_cv"
MAY_SPLIT       = ROOT / "data" / "metadata_unpaired_leakage_safe.csv"
MAY_PRED_DIRS   = {"Cytology ConvNeXt-Small": "safe_cyto_convnext_small", "Cytology Ensemble (v2)": "safe_cyto_ensemble_v2",
                   "Cytology ConvNeXt-Tiny": "safe_cyto_convnext", "Cytology Swin-Tiny": "safe_cyto_swin",
                   "Cytology EfficientNet-B3": "safe_cyto_efficientnet", "Ultrasound EfficientNet-B3": "safe_us_efficientnet"}
PRIOR_PAPER_CSV = Path(r"C:\Thyroid\metadata_with_holdout.csv")      # published 384-patient analysis (optional)

TN3K_ROOT          = ROOT / "Thyroid Dataset" / "tn3k"
DDTI_ORIGINAL_ROOT = ROOT / "Thyroid Dataset" / "DDTI_original"

ROUND2      = ROOT / "revision_round2"
MODELS      = ["convnext_small", "efficientnet_b3", "swin_tiny", "resnet50", "densenet121"]
DISPLAY     = {"convnext_small": "ConvNeXt-Small", "efficientnet_b3": "EfficientNet-B3", "swin_tiny": "Swin-Tiny",
               "resnet50": "ResNet-50", "densenet121": "DenseNet-121", "ensemble": "Ensemble (mean of 5)"}
BATCH_SIZE  = 16
NUM_WORKERS = 4
N_BOOT      = 2000
SEED        = 2026

RUN = dict(A=True, F=True, H=True, G=True, I=True, B=True, C=True, D=True, E=True, J=True, K=True)

## Setup and helpers (no edits needed)

In [ ]:
import os, sys, re, io, json, time, shutil, hashlib, zipfile, datetime, subprocess, contextlib, traceback, warnings, itertools
import xml.etree.ElementTree as ET
from collections import Counter
import numpy as np
import pandas as pd

def _ensure(pkgs):
    missing = []
    for mod, pipname in pkgs:
        try:
            __import__(mod)
        except ImportError:
            missing.append(pipname)
    if missing:
        print("installing into this kernel:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
_ensure([("sklearn", "scikit-learn"), ("scipy", "scipy"), ("matplotlib", "matplotlib"), ("PIL", "pillow")])

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
from PIL import Image, ImageFile
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

for sub in ["logs", "tables", "figures", "predictions", "manifests", "scripts", "external", "cv_innerval",
            "imagelevel_cv", "shortcut_cv", "gradcam", "embeddings", "release"]:
    (ROUND2 / sub).mkdir(parents=True, exist_ok=True)
RESULTS_MD = ROUND2 / "RESULTS.md"
if not RESULTS_MD.exists():
    RESULTS_MD.write_text("# Round-2 revision results\n", encoding="utf-8")
STATUS = {}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
                     "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False, "font.family": "DejaVu Sans"})

class _Tee(io.TextIOBase):
    def __init__(self, *s): self.s = s
    def write(self, x):
        for st in self.s: st.write(x)
        return len(x)
    def flush(self):
        for st in self.s: st.flush()

@contextlib.contextmanager
def stage(key, title):
    log = ROUND2 / "logs" / f"stage_{key}.log"
    with open(log, "a", encoding="utf-8") as fh, contextlib.redirect_stdout(_Tee(sys.stdout, fh)):
        t0 = time.time()
        print("\n" + "=" * 100 + f"\nSTAGE {key}: {title}   [{datetime.datetime.now():%Y-%m-%d %H:%M}]\n" + "=" * 100)
        if not RUN.get(key, True):
            print("skipped (RUN flag is False)"); STATUS[key] = "skipped"
            yield False
            return
        try:
            yield True
            STATUS[key] = f"done in {(time.time() - t0) / 60:.1f} min"
        except Exception:
            print("!!! STAGE FAILED — later stages continue. Traceback:\n" + traceback.format_exc())
            STATUS[key] = "FAILED (see log)"
        print(f"--- stage {key}: {STATUS.get(key)}")

def run_cmd(cmd, log_name, env_extra=None):
    """Run a command, streaming output live and to logs/<log_name>. Raises on non-zero exit."""
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    if env_extra: env.update(env_extra)
    log = ROUND2 / "logs" / log_name
    print("$ " + " ".join(str(c) for c in cmd) + (f"   [env {env_extra}]" if env_extra else ""))
    with open(log, "a", encoding="utf-8") as fh:
        fh.write(f"\n### {datetime.datetime.now():%Y-%m-%d %H:%M}  " + " ".join(str(c) for c in cmd) + "\n")
        p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", errors="replace", env=env)
        last = time.time()
        for line in p.stdout:
            fh.write(line); fh.flush()
            if ("epoch" in line and time.time() - last < 20) and "CV SUMMARY" not in line:
                continue  # keep the notebook readable; the full log has every line
            print(line.rstrip()); last = time.time()
        rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed (exit {rc}); see {log}")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def parts(p):
    from pathlib import PureWindowsPath, PurePosixPath
    s = str(p); return PureWindowsPath(s).parts if "\\" in s else PurePosixPath(s).parts

def case_key_from_path(p):
    q = [x.lower() for x in parts(p)]
    folder = " ".join(q[:-1])
    lab = "PTC" if ("papillary" in folder or "carcinoma" in folder) else ("BEN" if "benign" in folder else None)
    m = re.match(r"^(\d+)", Path(q[-1]).stem)
    return f"{lab}_{int(m.group(1)):04d}" if (lab and m) else None

def img_id(p):
    return "/".join(x.lower() for x in parts(p)[-2:])

def save_table(df, name, title, note=None, show=True):
    df.to_csv(ROUND2 / "tables" / f"{name}.csv", index=False)
    with open(RESULTS_MD, "a", encoding="utf-8") as f:
        f.write(f"\n\n## {name}: {title}\n")
        if note: f.write(f"\n{note}\n")
        f.write("\n" + df.to_markdown(index=False) + "\n" if hasattr(df, "to_markdown") and _has_tabulate() else "\n```\n" + df.to_string(index=False) + "\n```\n")
    if show:
        print(f"\n[{name}] {title}"); print(df.to_string(index=False))

def _has_tabulate():
    try:
        import tabulate  # noqa
        return True
    except ImportError:
        return False

def note_md(text):
    with open(RESULTS_MD, "a", encoding="utf-8") as f: f.write("\n" + text + "\n")
    print(text)

In [ ]:
# ------------------------------- statistics -------------------------------
def youden(y, p):
    """Identical to 03b.youden."""
    fpr, tpr, thr = roc_curve(np.asarray(y, int), np.asarray(p, float))
    finite = np.where(np.isfinite(thr))[0]
    if len(finite) == 0: return 0.5
    return float(thr[finite[np.argmax(tpr[finite] - fpr[finite])]])

def ece15(y, p, bins=15):
    """Expected calibration error: 15 equal-width bins on [0,1], weighted by bin count; bin i = (i/15, (i+1)/15]."""
    y = np.asarray(y, float); p = np.asarray(p, float)
    idx = np.clip(np.ceil(p * bins).astype(int) - 1, 0, bins - 1)
    return float(sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean()) for b in range(bins) if (idx == b).any()))

def op_metrics(y, p, thr):
    y = np.asarray(y, int); pr = (np.asarray(p, float) >= thr).astype(int)
    tp = int(((pr == 1) & (y == 1)).sum()); tn = int(((pr == 0) & (y == 0)).sum())
    fp = int(((pr == 1) & (y == 0)).sum()); fn = int(((pr == 0) & (y == 1)).sum())
    d = lambda a, b: a / b if b else np.nan
    return dict(threshold=float(thr), tn=tn, fp=fp, fn=fn, tp=tp, accuracy=d(tp + tn, len(y)), sensitivity=d(tp, tp + fn),
                specificity=d(tn, tn + fp), ppv=d(tp, tp + fp), npv=d(tn, tn + fn), f1=d(2 * tp, 2 * tp + fp + fn))

def full_metrics(y, p, thr=None):
    y = np.asarray(y, int); p = np.asarray(p, float); two = len(np.unique(y)) == 2
    out = dict(n=len(y), n_pos=int(y.sum()), n_neg=int((1 - y).sum()), prevalence=float(y.mean()) if len(y) else np.nan,
               auroc=roc_auc_score(y, p) if two else np.nan, auprc=average_precision_score(y, p) if two else np.nan,
               brier=float(np.mean((p - y) ** 2)), ece15=ece15(y, p))
    if thr is not None: out.update(op_metrics(y, p, thr))
    return out

def boot_ci(y, p, groups, thr=None, n_boot=None, seed=SEED):
    """Percentile 95% CIs from resampling whole groups (cases/components) with replacement."""
    n_boot = n_boot or N_BOOT
    y = np.asarray(y, int); p = np.asarray(p, float); g = np.asarray(groups).astype(str)
    uniq, inv = np.unique(g, return_inverse=True)
    members = [np.where(inv == k)[0] for k in range(len(uniq))]
    rng = np.random.default_rng(seed); rec = {k: [] for k in ["auroc", "auprc", "brier", "sensitivity", "specificity", "accuracy"]}
    for _ in range(n_boot):
        idx = np.concatenate([members[k] for k in rng.integers(0, len(uniq), len(uniq))])
        yy, pp = y[idx], p[idx]
        if len(np.unique(yy)) < 2: continue
        rec["auroc"].append(roc_auc_score(yy, pp)); rec["auprc"].append(average_precision_score(yy, pp))
        rec["brier"].append(np.mean((pp - yy) ** 2))
        if thr is not None:
            o = op_metrics(yy, pp, thr)
            for k in ("sensitivity", "specificity", "accuracy"): rec[k].append(o[k])
    return {k: (float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))) for k, v in rec.items() if v}

def ci_str(v, ci):
    return "NA" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.3f} ({ci[0]:.3f}–{ci[1]:.3f})"

def to_logit(p):
    p = np.clip(np.asarray(p, float), 1e-7, 1 - 1e-7); return np.log(p / (1 - p))

def fit_temperature(y, p):
    z = to_logit(p); y = np.asarray(y, float)
    nll = lambda T: -np.mean(y * np.log(1 / (1 + np.exp(-z / T)) + 1e-12) + (1 - y) * np.log(1 - 1 / (1 + np.exp(-z / T)) + 1e-12))
    return float(minimize_scalar(nll, bounds=(0.05, 100.0), method="bounded").x)

def apply_temperature(p, T):
    return 1 / (1 + np.exp(-to_logit(p) / T))

def reliability(y, p, bins=10):
    y = np.asarray(y, float); p = np.asarray(p, float); edges = np.linspace(0, 1, bins + 1); rows = []
    for i in range(bins):
        m = (p > edges[i]) & (p <= edges[i + 1]) if i else (p >= 0) & (p <= edges[1])
        if m.any(): rows.append(dict(bin=i, mean_pred=p[m].mean(), frac_pos=y[m].mean(), n=int(m.sum())))
    return pd.DataFrame(rows)

def near_binary_fraction(path):
    a = np.asarray(Image.open(path).convert("L"), dtype=np.uint8)
    return float(((a <= 10) | (a >= 245)).mean())
print("helpers ready | python", sys.version.split()[0])

## Preflight: paths, GPU environment, datasets, runtime plan

In [ ]:
with stage("PRE", "Preflight") as go:
    checks = []
    def chk(name, ok, detail=""):
        checks.append(dict(check=name, ok="yes" if ok else "NO", detail=str(detail))); return ok
    chk("project root", ROOT.exists(), ROOT)
    man_ok = chk("frozen manifest present", FROZEN_MANIFEST.exists(), FROZEN_MANIFEST)
    if man_ok:
        sha = sha256_file(FROZEN_MANIFEST); chk("frozen manifest SHA-256 matches", sha == FROZEN_SHA256, sha)
    chk("script 03b present", SCRIPT_03B.exists(), SCRIPT_03B)
    chk("exp1b checkpoints (ultrasound, 5)", all((EXP1B_DIR / "ultrasound" / k / "best_model.pt").exists() for k in MODELS))
    chk("exp1b predictions (both modalities)", all((EXP1B_DIR / mo / k / "test_predictions.csv").exists() for mo in ["cytology", "ultrasound"] for k in MODELS + ["ensemble"]))
    chk("May image-level split", MAY_SPLIT.exists(), MAY_SPLIT)
    torch_ok = chk("TORCH_PY exists", TORCH_PY.exists(), TORCH_PY)
    if torch_ok:
        r = subprocess.run([str(TORCH_PY), "-c", "import torch,timm,torchvision,sklearn,scipy;print(torch.__version__,timm.__version__,torch.cuda.is_available(),torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"],
                           capture_output=True, text=True)
        chk("torch/timm import in TORCH_PY", r.returncode == 0, (r.stdout or r.stderr).strip()[-200:])
        chk("CUDA available", "True" in r.stdout, r.stdout.strip())
    t_lab = list(TN3K_ROOT.glob("**/label4test.csv")) if TN3K_ROOT.exists() else []
    chk("TN3K label4test.csv (Stage H)", bool(t_lab), t_lab[0] if t_lab else f"not found under {TN3K_ROOT} — download from the TN3K authors")
    d_xml = list(DDTI_ORIGINAL_ROOT.rglob("*.xml")) if DDTI_ORIGINAL_ROOT.exists() else []
    chk("original DDTI XML files (Stage H)", bool(d_xml), f"{len(d_xml)} XML files" if d_xml else f"not found under {DDTI_ORIGINAL_ROOT} — download from Kaggle")
    done = lambda mo, k: (CV_DIR / mo / k / "cv_summary.json").exists()
    todo_cv = [(mo, k) for mo in ["ultrasound", "cytology"] for k in MODELS if not done(mo, k)]
    chk("grouped CV runs still to do (Stage B)", True, f"{len(todo_cv)} of 10: {todo_cv}")
    free_gb = shutil.disk_usage(str(ROOT)).free / 1e9
    chk("free disk on project drive (need ~15 GB for checkpoints)", free_gb > 15, f"{free_gb:.0f} GB")
    save_table(pd.DataFrame(checks), "PRE_checks", "Preflight checks")
    hours = {"convnext_small": 2.4, "efficientnet_b3": 1.2, "swin_tiny": 1.6, "resnet50": 1.2, "densenet121": 1.3}
    est_b = sum(hours[k] for _, k in todo_cv)
    plan = pd.DataFrame([dict(stage="B grouped CV", hours=round(est_b, 1)), dict(stage="D image-level CV (cytology, 2 models)", hours=3.6),
                         dict(stage="E shortcut control (ultrasound EfficientNet-B3, 2 masks)", hours=2.4),
                         dict(stage="C, F, G, H, I (inference/analysis)", hours=0.7)])
    plan.loc[len(plan)] = dict(stage="TOTAL (estimate from your August run times)", hours=round(plan.hours.sum(), 1))
    save_table(plan, "PRE_runtime_plan", "Estimated GPU time (resumable)")

## Scripts written by this notebook
- `r2_gpu_tasks.py` — inference only; imports your 03b.
- `03b_imagelevel_cv.py` — your 03b with **one line** changed: folds grouped by image.
- `03b_shortcut_cv.py` — your 03b plus a sector-mask transform.

In [ ]:
# Shared code (verified): sector detector, pHash identical to imagehash, patches to 03b, GPU inference script
SECTOR_SRC = r'''
# ---- sector box detector (shared by the notebook and the shortcut-control training script) ----
import hashlib as _hl
import numpy as _np
from PIL import Image as _Image
try:
    from scipy import ndimage as _ndi
except Exception:  # scipy is a scikit-learn dependency, so this should not happen
    _ndi = None

_SECTOR_CACHE = {}
SECTOR_FALLBACK = (0.12, 0.12, 0.88, 0.90)

def sector_box(img):
    """Bounding box (x0, y0, x1, y1 as fractions) of the largest textured region = the B-mode sector.
    Thin burned-in text and markers are removed by a 7x7 morphological opening.
    Falls back to a fixed central box when the detected region is implausibly small or large."""
    g = _np.asarray(img.convert("L"), dtype=_np.float32)
    h, w = g.shape
    key = (w, h, _hl.md5(_np.asarray(img.convert("L").resize((32, 32))).tobytes()).hexdigest())
    if key in _SECTOR_CACHE:
        return _SECTOR_CACHE[key]
    result = (SECTOR_FALLBACK, "fallback")
    if _ndi is not None and h > 20 and w > 20:
        m = _ndi.uniform_filter(g, 9)
        s = _np.sqrt(_np.clip(_ndi.uniform_filter(g * g, 9) - m * m, 0, None))
        tex = (s > 6.0) & (g > 12.0)
        tex = _ndi.binary_opening(tex, structure=_np.ones((7, 7), bool))
        lab, n = _ndi.label(tex)
        if n > 0:
            sizes = _ndi.sum(tex, lab, range(1, n + 1))
            k = int(_np.argmax(sizes)) + 1
            ys, xs = _np.where(lab == k)
            box = (xs.min() / w, ys.min() / h, (xs.max() + 1) / w, (ys.max() + 1) / h)
            area = (box[2] - box[0]) * (box[3] - box[1])
            if 0.25 <= area <= 0.97:
                result = (tuple(float(v) for v in box), "detected")
    _SECTOR_CACHE[key] = result
    return result

class SectorMask:
    """mode='surround_only' blacks out the sector; mode='center_only' blacks out everything outside it."""
    def __init__(self, mode):
        if mode not in ("surround_only", "center_only"):
            raise ValueError(mode)
        self.mode = mode
    def __call__(self, img):
        (x0, y0, x1, y1), _ = sector_box(img)
        arr = _np.array(img.convert("RGB"))
        h, w = arr.shape[:2]
        X0, Y0 = int(x0 * w), int(y0 * h)
        X1, Y1 = int(_np.ceil(x1 * w)), int(_np.ceil(y1 * h))
        if self.mode == "surround_only":
            out = arr.copy(); out[Y0:Y1, X0:X1] = 0
        else:
            out = _np.zeros_like(arr); out[Y0:Y1, X0:X1] = arr[Y0:Y1, X0:X1]
        return _Image.fromarray(out)
'''

PHASH_SRC = r'''
# ---- pHash identical to imagehash.phash (hash_size=8, highfreq_factor=4) ----
import numpy as _np2
from PIL import Image as _Image2
from scipy.fftpack import dct as _dct

def phash_hex(img):
    try:
        lanczos = _Image2.Resampling.LANCZOS
    except AttributeError:
        lanczos = _Image2.LANCZOS
    px = _np2.asarray(img.convert("L").resize((32, 32), lanczos), dtype=_np2.float64)
    d = _dct(_dct(px, axis=0), axis=1)[:8, :8]
    bits = (d > _np2.median(d)).flatten()
    return "{:016x}".format(int("".join("1" if b else "0" for b in bits), 2))

def hex_to_u64(h):
    return _np2.uint64(int(str(h), 16))

def hamming_matrix(a_hex, b_hex):
    a = _np2.array([int(str(x), 16) for x in a_hex], dtype=_np2.uint64)
    b = _np2.array([int(str(x), 16) for x in b_hex], dtype=_np2.uint64)
    x = _np2.bitwise_xor(a[:, None], b[None, :])
    return _np2.unpackbits(x.view(_np2.uint8).reshape(x.shape + (8,)), axis=-1).sum(-1).astype(_np2.int16)
'''

PATCH_SRC = r'''
def patch_03b_imagelevel(src):
    """Only change: the CV grouping key becomes the individual image instead of the duplicate-connected case component."""
    old = '+ "::component" + manifest["component_id"].astype(str)'
    new = '+ "::image" + manifest["image_path"].astype(str)  # ROUND2 PATCH: image-level folds (leakage demonstration)'
    if src.count(old) != 1:
        raise RuntimeError("03b grouping line not found exactly once; script version differs from the audited one")
    out = src.replace(old, new)
    out = out.replace('print("Grouping unit   : duplicate-connected component_id")',
                      'print("Grouping unit   : INDIVIDUAL IMAGE (round-2 leakage demonstration)")')
    return out

def patch_03b_shortcut(src, sector_src):
    """Only change: a SectorMask transform is prepended to the train and eval pipelines (mode from THYRO_MASK_MODE)."""
    anchor = "def transforms_for_run():"
    if src.count(anchor) != 1:
        raise RuntimeError("transforms_for_run() not found exactly once")
    old = "transforms.Compose([\n        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),"
    new = "transforms.Compose([\n        SectorMask(MASK_MODE),  # ROUND2 PATCH\n        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),"
    if src.count(old) != 2:
        raise RuntimeError(f"expected 2 Compose/Resize blocks, found {src.count(old)}")
    header = sector_src + '\nMASK_MODE = os.environ.get("THYRO_MASK_MODE", "")\n' \
             'if MASK_MODE not in ("surround_only", "center_only"):\n' \
             '    raise RuntimeError("Set THYRO_MASK_MODE to surround_only or center_only")\n\n\n'
    return src.replace(old, new).replace(anchor, header + anchor)
'''

GPU_TASKS_SRC = r'''
"""Round-2 GPU tasks (inference only; no training).
Imports the audited 03b script, so datasets, transforms, prediction and fold construction
are identical to the cross-validation runs that produced the checkpoints."""
import argparse, importlib.util, json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import timm
from PIL import Image


def load_module(path, name="exp2b"):
    spec = importlib.util.spec_from_file_location(name, str(path))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def tload(path, device):
    try:
        return torch.load(str(path), map_location=device, weights_only=False)
    except TypeError:
        return torch.load(str(path), map_location=device)


def device_of():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_model(m, key, ckpt, device):
    model = timm.create_model(m.MODELS[key], pretrained=False, num_classes=1)
    state = tload(ckpt, device)
    sd = state["model_state_dict"] if isinstance(state, dict) and "model_state_dict" in state else state
    model.load_state_dict(sd)
    return model.to(device).eval()


def free(model):
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def with_labels(df):
    df = df.reset_index(drop=True).copy()
    if "label" not in df.columns:
        df["label"] = -1
    df["label"] = pd.to_numeric(df["label"], errors="coerce").fillna(-1)
    return df


def cmd_cv_innerval(a):
    m = load_module(a.script03b)
    dev = device_of()
    manifest = pd.read_csv(a.manifest, low_memory=False)
    manifest["_component_uid"] = (manifest["modality"].astype(str).str.lower()
                                  + "::component" + manifest["component_id"].astype(str))
    devset = manifest[manifest["split"].isin(["train", "val"])].copy()
    mdev = devset[devset["modality"].astype(str).str.lower().eq(a.modality)].copy().reset_index(drop=True)
    outer = m.make_outer_splits(mdev, m.SEED)
    _, eval_tf = m.transforms_for_run()
    ref = None
    if a.fold_assignment and Path(a.fold_assignment).exists():
        ref = pd.read_csv(a.fold_assignment)
        ref = ref[ref["modality"].astype(str).str.lower().eq(a.modality)]
    for fold, (tr_idx, ho_idx) in enumerate(outer, 1):
        fold_seed = m.SEED + fold * 100
        outer_train = mdev.iloc[tr_idx].reset_index(drop=True)
        hold = mdev.iloc[ho_idx]
        if ref is not None and len(ref):
            r = set(ref.loc[ref["outer_fold"].eq(fold), "image_path"])
            if r != set(hold["image_path"]):
                raise RuntimeError(f"{a.modality} fold {fold}: replicated holdout differs from fold_assignment.csv")
        _, iv_idx = m.make_inner_split(outer_train, fold_seed + 1)
        inner_val = outer_train.iloc[iv_idx].reset_index(drop=True)
        loader = m.make_loader(inner_val, eval_tf, a.batch_size, 0)
        for key in a.models:
            dst = Path(a.out) / a.modality / key / f"fold_{fold}_innerval_predictions.csv"
            if dst.exists():
                print("[skip]", dst, flush=True)
                continue
            ckpt = Path(a.cv_dir) / a.modality / key / f"fold_{fold}" / "best_model.pt"
            if not ckpt.exists():
                raise FileNotFoundError(ckpt)
            model = build_model(m, key, ckpt, dev)
            pred = m.predict(model, loader, inner_val, dev)
            thr = float(m.youden(pred["label"], pred["prob_malignant"]))
            out = pred[["image_path", "label", "source_group", "component_id", "prob_malignant"]].copy()
            out["outer_fold"] = fold
            out["model_key"] = key
            out["recomputed_threshold"] = thr
            dst.parent.mkdir(parents=True, exist_ok=True)
            out.to_csv(dst, index=False)
            print(f"{a.modality} fold {fold} {key}: inner-val n={len(out)} threshold={thr:.6f}", flush=True)
            free(model)


def cmd_check_folds(a):
    """Exit 0 if 03b in THIS environment reproduces the saved fold assignment exactly (both modalities), else exit 3."""
    import sklearn
    m = load_module(a.script03b)
    manifest = pd.read_csv(a.manifest, low_memory=False)
    manifest["_component_uid"] = (manifest["modality"].astype(str).str.lower()
                                  + "::component" + manifest["component_id"].astype(str))
    devset = manifest[manifest["split"].isin(["train", "val"])]
    ref = pd.read_csv(a.fold_assignment)
    report = {}
    for mo in ["ultrasound", "cytology"]:
        r = ref[ref["modality"].astype(str).str.lower().eq(mo)]
        if r.empty:
            continue
        mdev = devset[devset["modality"].astype(str).str.lower().eq(mo)].copy().reset_index(drop=True)
        outer = m.make_outer_splits(mdev, m.SEED)
        report[mo] = all(set(r.loc[r["outer_fold"].eq(f), "image_path"]) == set(mdev.iloc[h]["image_path"])
                         for f, (_, h) in enumerate(outer, 1))
    ok = bool(report) and all(report.values())
    print(json.dumps({"sklearn": sklearn.__version__, "folds_reproduced": report, "ok": ok}), flush=True)
    sys.exit(0 if ok else 3)


def cmd_predict(a):
    m = load_module(a.script03b)
    dev = device_of()
    df = with_labels(pd.read_csv(a.manifest))
    _, eval_tf = m.transforms_for_run()
    loader = m.make_loader(df, eval_tf, a.batch_size, 0)
    outdir = Path(a.out)
    outdir.mkdir(parents=True, exist_ok=True)
    for key in a.models:
        dst = outdir / f"{key}_predictions.csv"
        if dst.exists():
            print("[skip]", dst, flush=True)
            continue
        ckpt = Path(a.exp1b_dir) / a.modality / key / "best_model.pt"
        if not ckpt.exists():
            raise FileNotFoundError(ckpt)
        t0 = time.time()
        model = build_model(m, key, ckpt, dev)
        pred = m.predict(model, loader, df, dev)
        out = df.copy()
        out["prob_malignant"] = pred["prob_malignant"].values
        out["model_key"] = key
        out.to_csv(dst, index=False)
        print(f"{Path(a.manifest).name} {key}: n={len(out)} in {time.time() - t0:.0f}s", flush=True)
        free(model)


def cmd_embed(a):
    m = load_module(a.script03b)
    dev = device_of()
    df = with_labels(pd.read_csv(a.manifest))
    outdir = Path(a.out)
    outdir.mkdir(parents=True, exist_ok=True)
    ckpt = Path(a.exp1b_dir) / a.modality / a.model / "best_model.pt"
    model = build_model(m, a.model, ckpt, dev)
    model.reset_classifier(0)
    _, eval_tf = m.transforms_for_run()
    loader = m.make_loader(df, eval_tf, a.batch_size, 0)
    feats = None
    with torch.no_grad():
        for images, _, idx in loader:
            f = model(images.to(dev)).float().cpu().numpy()
            if feats is None:
                feats = np.zeros((len(df), f.shape[1]), np.float32)
            feats[idx.numpy()] = f
    np.save(outdir / "embeddings.npy", feats)
    df.to_csv(outdir / "embeddings_index.csv", index=False)
    print("embeddings:", feats.shape, flush=True)


def cmd_gradcam(a):
    m = load_module(a.script03b)
    dev = device_of()
    df = pd.read_csv(a.manifest).reset_index(drop=True)
    outdir = Path(a.out)
    outdir.mkdir(parents=True, exist_ok=True)
    ckpt = Path(a.exp1b_dir) / a.modality / a.model / "best_model.pt"
    model = build_model(m, a.model, ckpt, dev)
    modules = dict(model.named_modules())
    if a.layer not in modules:
        raise KeyError(f"layer {a.layer} not in model; candidates end with: {list(modules)[-8:]}")
    store = {}

    def hook(mod, inp, out):
        store["act"] = out
        out.register_hook(lambda g: store.__setitem__("grad", g))

    handle = modules[a.layer].register_forward_hook(hook)
    _, eval_tf = m.transforms_for_run()
    S = 224
    cams = np.zeros((len(df), S, S), np.float16)
    rows = []
    for i, r in df.iterrows():
        rec = {"image_path": r["image_path"]}
        try:
            with Image.open(r["image_path"]) as im:
                x = eval_tf(im.convert("RGB")).unsqueeze(0).to(dev)
            model.zero_grad(set_to_none=True)
            logit = model(x).reshape(-1)[0]
            logit.backward()
            A, G = store["act"].detach(), store["grad"].detach()
            cam = torch.relu((G.mean(dim=(2, 3), keepdim=True) * A).sum(1, keepdim=True))
            cam = torch.nn.functional.interpolate(cam.float(), size=(S, S), mode="bilinear", align_corners=False)[0, 0]
            cam = cam.cpu().numpy()
            X0, Y0 = int(r["box_x0"] * S), int(r["box_y0"] * S)
            X1, Y1 = int(np.ceil(r["box_x1"] * S)), int(np.ceil(r["box_y1"] * S))
            total = float(cam.sum())
            inside = float(cam[Y0:Y1, X0:X1].sum())
            py, px = np.unravel_index(int(cam.argmax()), cam.shape)
            rec.update(prob_malignant=float(torch.sigmoid(logit).item()),
                       cam_mass_outside_sector=(1 - inside / total) if total > 0 else np.nan,
                       sector_area_outside=1 - ((X1 - X0) * (Y1 - Y0)) / float(S * S),
                       peak_outside_sector=not (X0 <= px < X1 and Y0 <= py < Y1),
                       status="ok")
            cams[i] = (cam / cam.max() if cam.max() > 0 else cam).astype(np.float16)
        except Exception as e:
            rec.update(status=f"error: {e}")
        rows.append(rec)
    handle.remove()
    np.savez_compressed(outdir / "cams.npz", cams=cams)
    pd.DataFrame(rows).to_csv(outdir / "gradcam_per_image.csv", index=False)
    print("grad-cam images:", len(rows), "errors:", sum(1 for r in rows if r["status"] != "ok"), flush=True)


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("cv-innerval")
    p.add_argument("--script03b", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--cv-dir", required=True); p.add_argument("--fold-assignment", default="")
    p.add_argument("--modality", required=True); p.add_argument("--models", nargs="+", required=True)
    p.add_argument("--out", required=True); p.add_argument("--batch-size", type=int, default=16)
    p = sub.add_parser("check-folds")
    p.add_argument("--script03b", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--fold-assignment", required=True)
    p = sub.add_parser("predict")
    p.add_argument("--script03b", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--exp1b-dir", required=True); p.add_argument("--modality", default="ultrasound")
    p.add_argument("--models", nargs="+", required=True); p.add_argument("--out", required=True)
    p.add_argument("--batch-size", type=int, default=16)
    p = sub.add_parser("embed")
    p.add_argument("--script03b", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--exp1b-dir", required=True); p.add_argument("--modality", default="ultrasound")
    p.add_argument("--model", default="efficientnet_b3"); p.add_argument("--out", required=True)
    p.add_argument("--batch-size", type=int, default=16)
    p = sub.add_parser("gradcam")
    p.add_argument("--script03b", required=True); p.add_argument("--manifest", required=True)
    p.add_argument("--exp1b-dir", required=True); p.add_argument("--modality", default="ultrasound")
    p.add_argument("--model", default="efficientnet_b3"); p.add_argument("--layer", default="conv_head")
    p.add_argument("--out", required=True)
    a = ap.parse_args()
    {"cv-innerval": cmd_cv_innerval, "check-folds": cmd_check_folds, "predict": cmd_predict, "embed": cmd_embed, "gradcam": cmd_gradcam}[a.cmd](a)


if __name__ == "__main__":
    main()
'''


exec(SECTOR_SRC); exec(PHASH_SRC); exec(PATCH_SRC)
GPU_TASKS = ROUND2 / "scripts" / "r2_gpu_tasks.py"
GPU_TASKS.write_text(GPU_TASKS_SRC, encoding="utf-8")
PATCHED_IMAGELEVEL = ROUND2 / "scripts" / "03b_imagelevel_cv.py"
PATCHED_SHORTCUT = ROUND2 / "scripts" / "03b_shortcut_cv.py"
if SCRIPT_03B.exists():
    _src = SCRIPT_03B.read_text(encoding="utf-8")
    PATCHED_IMAGELEVEL.write_text(patch_03b_imagelevel(_src), encoding="utf-8")
    PATCHED_SHORTCUT.write_text(patch_03b_shortcut(_src, SECTOR_SRC), encoding="utf-8")
    print("wrote patched scripts:", PATCHED_IMAGELEVEL.name, PATCHED_SHORTCUT.name)
print("wrote", GPU_TASKS)

## Stage A — Provenance: cohort, duplicates, label conflicts, leakage, prior-paper reconciliation

In [ ]:
with stage("A", "Provenance") as go:
  if go:
    fm = pd.read_csv(FROZEN_MANIFEST, low_memory=False)
    may = pd.read_csv(MAY_SPLIT, low_memory=False)
    for d in (fm, may):
        d["case_key"] = d["image_path"].map(case_key_from_path); d["id"] = d["image_path"].map(img_id)

    # A1 archive composition
    rows = []
    for mo, d in may.groupby("modality"):
        rows.append(dict(modality=mo, images=len(d), benign_images=int((d.label == 0).sum()), ptc_images=int((d.label == 1).sum()),
                         cases=d.case_key.nunique(), benign_cases=d[d.label == 0].case_key.nunique(), ptc_cases=d[d.label == 1].case_key.nunique()))
    save_table(pd.DataFrame(rows), "A1_archive", "Pang archive as released (all files)")
    blocks = may[may.modality == "cytology"].groupby("case_key").size()
    us_cases = set(may[may.modality == "ultrasound"].case_key); cy_cases = set(may[may.modality == "cytology"].case_key)
    note_md(f"- Cytology blocks per case: {dict(sorted(Counter(blocks).items()))} (mean {blocks.mean():.2f}); "
            f"cases with both modalities: {len(us_cases & cy_cases)}; ultrasound-only cases: {sorted(us_cases - cy_cases)}; "
            f"case numbers reused across the two class folders: {int((may.assign(num=may.case_key.str[4:]).groupby('num').label.nunique() > 1).sum())}.")

    # A2 exact duplicates (MD5) and their handling in the frozen manifest
    cnt = may.md5.value_counts(); dup = may[may.md5.map(cnt) > 1].copy()
    dup["conflict_type"] = np.where(dup.md5.map(dup.groupby("md5").label.nunique()) > 1, "cross-label",
                                    np.where(dup.md5.map(dup.groupby("md5").case_key.nunique()) > 1, "same label, different cases", "within case"))
    dup["in_frozen_manifest"] = dup.id.isin(set(fm.id))
    dup["frozen_split"] = dup.id.map(dict(zip(fm.id, fm.split)))
    dup["may_split"] = dup["split"]
    save_table(dup.sort_values(["modality", "conflict_type", "md5", "case_key"])[["modality", "conflict_type", "case_key", "label", "id", "md5", "may_split", "in_frozen_manifest", "frozen_split"]],
               "A2_exact_duplicates", "Exact (MD5) duplicate groups, including label conflicts", show=False)
    summ = dup.groupby(["modality", "conflict_type"]).agg(groups=("md5", "nunique"), images=("id", "size"),
                                                          kept_in_frozen=("in_frozen_manifest", "sum")).reset_index()
    save_table(summ, "A2b_duplicate_summary", "Duplicate groups by type")
    if EXCLUDED_CSV.exists():
        ex = pd.read_csv(EXCLUDED_CSV)
        rc = "exclusion_reason" if "exclusion_reason" in ex.columns else ex.columns[-1]
        save_table(ex.groupby(rc).size().reset_index(name="images"), "A3_exclusions", "Images excluded from the frozen manifest")

    # A4 pHash near-duplicate sensitivity (Hamming distance, within modality)
    def pair_table(d):
        hx = d.phash.astype(str).tolist(); n = len(hx); out = []
        for i0 in range(0, n, 256):
            H = hamming_matrix(hx[i0:i0 + 256], hx)
            ii, jj = np.where(H <= 10)
            keep = jj > (ii + i0)
            for a, b in zip(ii[keep] + i0, jj[keep]):
                out.append((a, b, int(H[a - i0, b])))
        return pd.DataFrame(out, columns=["i", "j", "hamming"])
    sens_rows, sheet_pairs = [], []
    for mo, d in may.groupby("modality"):
        d = d.reset_index(drop=True); pt = pair_table(d)
        pt["cross_case"] = d.case_key.values[pt.i] != d.case_key.values[pt.j]
        pt["cross_label"] = d.label.values[pt.i] != d.label.values[pt.j]
        fsplit = d.id.map(dict(zip(fm.id, fm.split)))
        pt["both_in_frozen"] = fsplit.notna().values[pt.i] & fsplit.notna().values[pt.j]
        pt["cross_frozen_partition"] = pt.both_in_frozen & (fsplit.values[pt.i] != fsplit.values[pt.j])
        for t in [0, 2, 4, 6, 8, 10]:
            s = pt[pt.hamming <= t]
            sens_rows.append(dict(modality=mo, hamming_max=t, pairs=len(s), cross_case=int(s.cross_case.sum()), cross_label=int(s.cross_label.sum()),
                                  cross_frozen_partition=int(s.cross_frozen_partition.sum())))
        review = pt[pt.cross_case & (pt.hamming > 0) & (pt.hamming <= 6)].sort_values(["cross_frozen_partition", "hamming"], ascending=[False, True])
        pt.to_csv(ROUND2 / "tables" / f"A4b_phash_pairs_{mo}.csv", index=False)
        for _, r in review.head(6).iterrows():
            sheet_pairs.append((mo, d.image_path[r.i], d.image_path[r.j], d.case_key[r.i], d.case_key[r.j], r.hamming))
    save_table(pd.DataFrame(sens_rows), "A4_phash_sensitivity",
               "Perceptual-hash (DCT pHash, 64-bit) near-duplicate pairs by Hamming threshold",
               note="The original pipeline grouped identical pHash values only (Hamming 0). Pairs crossing frozen partitions are listed for visual review.")
    if sheet_pairs:
        fig, axes = plt.subplots(len(sheet_pairs), 2, figsize=(5.2, 2.2 * len(sheet_pairs)))
        axes = np.atleast_2d(axes)
        for k, (mo, a, b, ca, cb, h) in enumerate(sheet_pairs):
            for c, (pth, ck) in enumerate([(a, ca), (b, cb)]):
                ax = axes[k, c]; ax.imshow(Image.open(pth).convert("L"), cmap="gray"); ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(f"{mo} {ck}" + (f"  (Hamming {h})" if c == 0 else ""), fontsize=7)
        fig.suptitle("Non-identical near-duplicate candidates (Hamming 1–6), partition-crossing pairs first — for visual review", fontsize=7.5)
        fig.tight_layout(); fig.savefig(ROUND2 / "figures" / "S2_phash_near_duplicate_pairs.png"); plt.close(fig)

    # A5 measured leakage in the original (May) image-level split
    lk = []
    for mo, d in may.groupby("modality"):
        tr = set(d[d.split == "train"].case_key)
        for part in ["val", "test"]:
            t = d[d.split == part]
            lk.append(dict(modality=mo, partition=part, images=len(t), cases=t.case_key.nunique(),
                           images_with_same_case_in_train=round(float(t.case_key.isin(tr).mean()), 3),
                           cases_with_images_in_train=round(float(pd.Series(sorted(set(t.case_key))).isin(tr).mean()), 3),
                           cross_label_conflict_images=int(t.id.isin(set(dup[dup.conflict_type == "cross-label"].id)).sum())))
    save_table(pd.DataFrame(lk), "A5_leakage_original_split", "Measured case-level leakage in the original image-level split (Tables 3–13 of the submitted manuscript)")

    # A6 results as originally reported (image-level split), for the leakage comparison
    mr = []
    for name, folder in MAY_PRED_DIRS.items():
        f = ROOT / "outputs" / folder / "test_predictions.csv"
        if f.exists():
            p = pd.read_csv(f); m = full_metrics(p["label"].astype(int), p["prob_malignant"])
            mr.append(dict(model=name, n=m["n"], auroc=round(m["auroc"], 4), auprc=round(m["auprc"], 4), brier=round(m["brier"], 4)))
    if mr: save_table(pd.DataFrame(mr), "A6_original_image_level_results", "Original (leaky, image-level) test results, recomputed from files")

    # A7 reconciliation with the published 384-patient analysis
    if PRIOR_PAPER_CSV.exists():
        pp = pd.read_csv(PRIOR_PAPER_CSV)
        pp["case_key"] = pp.apply(lambda r: f"{'PTC' if int(r.label) == 1 else 'BEN'}_{int(r.original_id):04d}", axis=1)
        us_md5 = may[may.modality == "ultrasound"].groupby("case_key").md5.first()
        pp["us_md5"] = pp.case_key.map(us_md5)
        conf_cases = set(dup[(dup.conflict_type == "cross-label")].case_key)
        dev_md5 = set(pp[pp.split != "holdout"].us_md5)
        ho = pp[pp.split == "holdout"]
        rows = [dict(item="patients", value=len(pp)), dict(item="benign / PTC", value=f"{int((pp.label == 0).sum())} / {int((pp.label == 1).sum())}"),
                dict(item="linkage", value="class folder + case number (ultrasound <n>.jpg <-> cytology <n>_<block>.tif)"),
                dict(item="split", value=str(dict(Counter(pp.split)))),
                dict(item="patients carrying a cross-label duplicated ultrasound image", value=int(pp.case_key.isin(conf_cases).sum())),
                dict(item="holdout patients whose ultrasound image is byte-identical to a development patient's", value=int(ho.us_md5.isin(dev_md5).sum())),
                dict(item="holdout patients involved", value=", ".join(sorted(ho[ho.us_md5.isin(dev_md5)].case_key)))]
        save_table(pd.DataFrame(rows), "A7_prior_paper_reconciliation", "Reconciliation with the published BMC Medical Imaging analysis")

    flow = dict(archive_files=int(len(may)), us_images=int((may.modality == "ultrasound").sum()), cyto_blocks=int((may.modality == "cytology").sum()),
                us_cases=len(us_cases), cyto_cases=len(cy_cases),
                excluded_us=int(len(may[may.modality == "ultrasound"]) - (fm.modality == "ultrasound").sum()),
                frozen=fm.groupby(["modality", "split"]).agg(images=("image_path", "size"), cases=("source_group", "nunique"),
                                                             benign=("label", lambda s: int((s == 0).sum())), ptc=("label", "sum")).reset_index().to_dict("records"))
    (ROUND2 / "tables" / "A_flow.json").write_text(json.dumps(flow, indent=2, default=int), encoding="utf-8")
    save_table(pd.DataFrame(flow["frozen"]), "A8_frozen_partitions", "Frozen case-level partitions used for all new analyses")

## Stage F — Frozen test set: all backbones and ensemble, calibration

In [ ]:
with stage("F", "Frozen test-set evaluation (exp1b)") as go:
  if go:
    rows, calib, rel = [], [], []
    for mo in ["ultrasound", "cytology"]:
        for key in MODELS + ["ensemble"]:
            d = EXP1B_DIR / mo / key
            mj = json.loads((d / "metrics.json").read_text(encoding="utf-8"))
            for level, stem, thr_key in [("image", "", "image_level_threshold_from_val"), ("case", "_source_group", "source_group_threshold_from_val")]:
                val = pd.read_csv(d / f"val{stem}_predictions.csv"); test = pd.read_csv(d / f"test{stem}_predictions.csv")
                thr = float(mj[thr_key])
                assert abs(youden(val.label, val.prob_malignant) - thr) < 1e-6, f"{mo}/{key}/{level}: threshold is not the validation Youden point"
                m = full_metrics(test.label, test.prob_malignant, thr)
                ci = boot_ci(test.label, test.prob_malignant, test.component_id, thr)
                rows.append(dict(modality=mo, model=DISPLAY[key], level=level, n=m["n"], benign=m["n_neg"], ptc=m["n_pos"],
                                 val_auroc=round(roc_auc_score(val.label, val.prob_malignant), 3),
                                 auroc=ci_str(m["auroc"], ci["auroc"]), auprc=ci_str(m["auprc"], ci["auprc"]), no_skill_auprc=round(m["prevalence"], 3),
                                 threshold_from_val=round(thr, 4), tn=m["tn"], fp=m["fp"], fn=m["fn"], tp=m["tp"],
                                 sensitivity=ci_str(m["sensitivity"], ci["sensitivity"]), specificity=ci_str(m["specificity"], ci["specificity"]),
                                 accuracy=ci_str(m["accuracy"], ci["accuracy"]), f1=round(m["f1"], 3), brier=round(m["brier"], 3), ece15=round(m["ece15"], 3)))
                if level == "image":
                    T = fit_temperature(val.label, val.prob_malignant); T_at_bound = T > 99
                    pv, pt = apply_temperature(val.prob_malignant, T), apply_temperature(test.prob_malignant, T)
                    thr_scaled = youden(val.label, pv)
                    same_numeric = op_metrics(test.label, pt, thr); reselected = op_metrics(test.label, pt, thr_scaled)
                    orig = op_metrics(test.label, test.prob_malignant, thr)
                    calib.append(dict(modality=mo, model=DISPLAY[key], temperature=(">100 (search bound; near-uninformative logits)" if T_at_bound else round(T, 3)),
                                      test_ece_before=round(ece15(test.label, test.prob_malignant), 3), test_ece_after=round(ece15(test.label, pt), 3),
                                      test_brier_before=round(float(np.mean((test.prob_malignant - test.label) ** 2)), 3),
                                      test_brier_after=round(float(np.mean((pt - test.label) ** 2)), 3),
                                      accuracy_original=round(orig["accuracy"], 3), accuracy_scaled_same_numeric_threshold=round(same_numeric["accuracy"], 3),
                                      accuracy_scaled_threshold_reselected_on_val=round(reselected["accuracy"], 3),
                                      decisions_identical_after_reselection=bool((orig["tp"], orig["fp"]) == (reselected["tp"], reselected["fp"]))))
                    if key == "ensemble":
                        for tag, pp in [("before", test.prob_malignant.values), ("after", pt)]:
                            r = reliability(test.label, pp); r["modality"] = mo; r["scaling"] = tag; rel.append(r)
    F = pd.DataFrame(rows)
    save_table(F[F.level == "image"], "F1_frozen_test_image_level", "Frozen test set, image level; thresholds selected on validation; case-clustered 95% CIs")
    save_table(F[F.level == "case"], "F2_frozen_test_case_level", "Frozen test set, case level (images aggregated per case); thresholds from validation")
    save_table(pd.DataFrame(calib), "F3_temperature_scaling", "Temperature scaling fitted on validation; test ECE (15 equal-width bins) and decisions",
               note="Temperature scaling is monotone: re-selecting the threshold on validation after scaling leaves every decision unchanged; reusing the old numeric threshold does not.")
    pd.concat(rel).to_csv(ROUND2 / "tables" / "F4_reliability_source.csv", index=False)

## Stage H — External validation, done correctly
**Pre-specified before any external prediction is examined:**
- TN3K positive class = label 1 (malignant) from `label4test.csv`.
- DDTI primary endpoint: TI-RADS 2–3 (negative) vs 4–5 (positive), using the `<tirads>` field of the original XML.
- Sensitivity endpoints: 2–4a vs 4b–5, and 2 vs 4–5.
- Models are the frozen exp1b checkpoints; thresholds come from internal validation. **No label direction is ever changed.**

In [ ]:
def read_label_csv(path):
    raw = pd.read_csv(path, header=None, dtype=str)
    lab_col = next((c for c in raw.columns if raw[c].iloc[1:].str.strip().isin(["0", "1"]).mean() > 0.9), None)
    if lab_col is None: raise ValueError(f"no 0/1 column in {path}")
    if raw[lab_col].iloc[0].strip() not in ("0", "1"): raw = raw.iloc[1:]
    name_col = next(c for c in raw.columns if c != lab_col)
    out = pd.DataFrame({"name": raw[name_col].str.strip(), "label": raw[lab_col].str.strip().astype(int)})
    out["stem"] = out["name"].map(lambda s: Path(s).stem)
    return out

def build_tn3k(split):
    img_dir = TN3K_ROOT / f"{split}-image"; lab_files = list(TN3K_ROOT.rglob(f"label4{split}.csv"))
    if not img_dir.exists() or not lab_files:
        print(f"TN3K {split}: missing {'images' if not img_dir.exists() else 'label4' + split + '.csv'} -> skipped"); return None
    lab = read_label_csv(lab_files[0]); files = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() in (".jpg", ".png", ".jpeg", ".bmp")}
    lab["image_path"] = lab.stem.map(lambda s: str(files[s]) if s in files else (str(files[str(int(s)).zfill(4)]) if s.isdigit() and str(int(s)).zfill(4) in files else None))
    miss = int(lab.image_path.isna().sum())
    lab = lab.dropna(subset=["image_path"]).reset_index(drop=True)
    nb = lab.image_path.map(near_binary_fraction)
    n_mask = int((nb >= 0.97).sum())
    if n_mask: raise RuntimeError(f"TN3K {split}: {n_mask} mask-like inputs found — refusing to evaluate")
    lab["dataset"] = f"TN3K_{split}"; lab["case_id"] = lab.stem
    note_md(f"- TN3K {split}: {len(lab)} images from {lab_files[0].name} ({dict(Counter(lab.label))}; 0 = benign, 1 = malignant); "
            f"label rows without an image: {miss}; mask-like inputs: {n_mask}; image files without a label: {len(files) - len(lab)}.")
    return lab[["image_path", "label", "dataset", "case_id"]]

def build_ddti():
    if not DDTI_ORIGINAL_ROOT.exists():
        print("DDTI original: folder missing -> skipped"); return None
    xmls = sorted(DDTI_ORIGINAL_ROOT.rglob("*.xml"))
    imgs = {}
    for p in DDTI_ORIGINAL_ROOT.rglob("*"):
        m = re.match(r"^(\d+)_(\d+)$", p.stem)
        if m and p.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"): imgs[(int(m.group(1)), int(m.group(2)))] = p
    cases, bad = [], []
    for x in xmls:
        try:
            root = ET.parse(x).getroot()
            num = root.findtext("number"); num = int(num) if num and num.strip().isdigit() else int(re.sub(r"\D", "", x.stem))
            tir = (root.findtext("tirads") or "").strip().lower()
            marks = [int(mk.findtext("image")) for mk in root.findall("mark") if (mk.findtext("image") or "").strip().isdigit()]
            cases.append(dict(case_id=num, tirads=tir, marked_images=marks))
        except Exception as e:
            bad.append((x.name, str(e)))
    rows = []
    for c in cases:
        for (num, k), p in imgs.items():
            if num == c["case_id"]: rows.append(dict(image_path=str(p), case_id=c["case_id"], tirads=c["tirads"]))
    df = pd.DataFrame(rows)
    if df.empty:
        print("DDTI original: no images matched XML cases -> skipped"); return None
    POS = {"4", "4a", "4b", "4c", "5"}
    df["y_primary"] = np.where(df.tirads.isin(["2", "3"]), 0, np.where(df.tirads.isin(POS), 1, np.nan))
    df["y_sens_4a_negative"] = np.where(df.tirads.isin(["2", "3", "4a"]), 0, np.where(df.tirads.isin(["4b", "4c", "5"]), 1, np.nan))
    df["y_sens_exclude_3"] = np.where(df.tirads.isin(["2"]), 0, np.where(df.tirads.isin(POS), 1, np.nan))
    df["label"] = df.y_primary.fillna(-1).astype(int); df["dataset"] = "DDTI_original"
    df["tirads"] = df["tirads"].replace("", "(missing)")
    tab = df.groupby("tirads").agg(images=("image_path", "size"), cases=("case_id", "nunique")).reset_index()
    save_table(tab, "H0_ddti_tirads_distribution", "Original DDTI: TI-RADS field (<tirads>) by images and cases",
               note=f"XML files: {len(xmls)}; unparseable: {len(bad)}; image files: {len(imgs)}; images without a parseable XML case: {len(imgs) - len(df)}. "
                    "Values outside 2–5 or empty are excluded from every endpoint.")
    return df

with stage("H", "External validation (official TN3K labels; original DDTI TI-RADS)") as go:
  if go:
    ext = {}
    for split in ["test", "trainval"]:
        t = build_tn3k(split)
        if t is not None: ext[f"TN3K_{split}"] = t
    dd = build_ddti()
    if dd is not None: ext["DDTI_original"] = dd
    if not ext:
        raise RuntimeError("No external dataset available. Download TN3K labels and/or the original DDTI (see top of notebook).")
    for name, d in ext.items():
        d.to_csv(ROUND2 / "manifests" / f"external_{name}.csv", index=False)

    # H1 cross-dataset near-duplicate screen (pHash identical to the internal pipeline)
    may = pd.read_csv(MAY_SPLIT, low_memory=False)
    pang_us = may[may.modality == "ultrasound"].reset_index(drop=True)
    scr = []
    for name, d in ext.items():
        cache = ROUND2 / "manifests" / f"phash_{name}.csv"
        if cache.exists(): hx = pd.read_csv(cache)
        else:
            hx = pd.DataFrame({"image_path": d.image_path, "phash": [phash_hex(Image.open(p)) for p in d.image_path]}); hx.to_csv(cache, index=False)
        mins = np.concatenate([hamming_matrix(hx.phash.tolist()[i:i + 512], pang_us.phash.astype(str).tolist()).min(1) for i in range(0, len(hx), 512)])
        scr.append(dict(external=name, images=len(hx), min_hamming_to_pang=int(mins.min()), n_le_4=int((mins <= 4).sum()), n_le_8=int((mins <= 8).sum()), n_le_10=int((mins <= 10).sum())))
    save_table(pd.DataFrame(scr), "H1_cross_dataset_duplicates", "Cross-dataset near-duplicate screen against the Pang ultrasound images (pHash Hamming)")

    # H2 frozen-model inference (GPU, resumable)
    for name, d in ext.items():
        run_cmd([TORCH_PY, GPU_TASKS, "predict", "--script03b", SCRIPT_03B, "--manifest", ROUND2 / "manifests" / f"external_{name}.csv",
                 "--exp1b-dir", EXP1B_DIR, "--modality", "ultrasound", "--models", *MODELS, "--out", ROUND2 / "external" / name,
                 "--batch-size", BATCH_SIZE], f"H_predict_{name}.log")

    # H3 metrics at internal-validation thresholds; no label flipping
    thr = {k: float(json.loads((EXP1B_DIR / "ultrasound" / k / "metrics.json").read_text())["image_level_threshold_from_val"]) for k in MODELS + ["ensemble"]}
    rows = []
    for name, d in ext.items():
        merged = d.copy()
        for k in MODELS:
            p = pd.read_csv(ROUND2 / "external" / name / f"{k}_predictions.csv")[["image_path", "prob_malignant"]].rename(columns={"prob_malignant": f"prob_{k}"})
            assert p.image_path.is_unique and len(p) == len(d), f"{name}/{k}: prediction file does not match manifest"
            merged = merged.merge(p, on="image_path", how="left", validate="one_to_one")
        assert merged[[f"prob_{k}" for k in MODELS]].notna().all().all()
        merged["prob_ensemble"] = merged[[f"prob_{k}" for k in MODELS]].mean(axis=1)
        merged.to_csv(ROUND2 / "predictions" / f"external_{name}.csv", index=False)
        endpoints = [("benign vs malignant (distributed label)", "label")] if name.startswith("TN3K") else \
                    [("TI-RADS 2–3 vs 4–5 (primary)", "y_primary"), ("TI-RADS 2–4a vs 4b–5", "y_sens_4a_negative"), ("TI-RADS 2 vs 4–5", "y_sens_exclude_3")]
        for ep_name, col in endpoints:
            e = merged[merged[col].isin([0, 1])] if col != "label" else merged
            groups = e.case_id.astype(str)
            for k in MODELS + ["ensemble"]:
                y, p = e[col].astype(int).values, e[f"prob_{k}"].values
                m = full_metrics(y, p, thr[k]); ci = boot_ci(y, p, groups, thr[k])
                rows.append(dict(dataset=name, endpoint=ep_name, model=DISPLAY[k], images=m["n"], cases=int(groups.nunique()),
                                 negatives=m["n_neg"], positives=m["n_pos"], auroc=ci_str(m["auroc"], ci["auroc"]),
                                 auprc=ci_str(m["auprc"], ci["auprc"]), no_skill_auprc=round(m["prevalence"], 3),
                                 threshold_internal=round(thr[k], 4), sensitivity=ci_str(m["sensitivity"], ci["sensitivity"]),
                                 specificity=ci_str(m["specificity"], ci["specificity"]), accuracy=round(m["accuracy"], 3),
                                 brier=round(m["brier"], 3), ece15=round(m["ece15"], 3),
                                 cluster_unit="case (XML)" if name.startswith("DDTI") else "image (no patient IDs distributed)"))
    save_table(pd.DataFrame(rows), "H2_external_results", "External evaluation of frozen ultrasound models (label direction pre-specified; never flipped)")

## Stage G — Grad-CAM with quantified attribution outside the ultrasound sector

In [ ]:
with stage("G", "Grad-CAM (exp1b ultrasound EfficientNet-B3)") as go:
  if go:
    key = "efficientnet_b3"; d = EXP1B_DIR / "ultrasound" / key
    thr = float(json.loads((d / "metrics.json").read_text())["image_level_threshold_from_val"])
    frames = []
    for part in ["val", "test"]:
        p = pd.read_csv(d / f"{part}_predictions.csv")[["image_path", "label", "prob_malignant"]]; p["partition"] = part; frames.append(p)
    gm = pd.concat(frames, ignore_index=True)
    boxes = [sector_box(Image.open(pth)) for pth in gm.image_path]
    gm[["box_x0", "box_y0", "box_x1", "box_y1"]] = pd.DataFrame([b[0] for b in boxes]); gm["box_method"] = [b[1] for b in boxes]
    gm.to_csv(ROUND2 / "manifests" / "gradcam_manifest.csv", index=False)
    note_md(f"- Sector box detection on {len(gm)} ultrasound val/test images: {dict(Counter(gm.box_method))}.")
    # QA sheet of detected sector boxes
    qa = gm.sample(min(24, len(gm)), random_state=SEED)
    fig, axes = plt.subplots(4, 6, figsize=(9, 6.2))
    for ax, (_, r) in zip(axes.ravel(), qa.iterrows()):
        im = Image.open(r.image_path).convert("L"); w, h = im.size; ax.imshow(im, cmap="gray")
        ax.add_patch(Rectangle((r.box_x0 * w, r.box_y0 * h), (r.box_x1 - r.box_x0) * w, (r.box_y1 - r.box_y0) * h, fill=False, ec="#f5a623", lw=1.2))
        ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"{case_key_from_path(r.image_path)} ({r.box_method})", fontsize=6)
    for ax in axes.ravel()[len(qa):]: ax.axis("off")
    fig.tight_layout(); fig.savefig(ROUND2 / "figures" / "S3_sector_box_QA.png"); plt.close(fig)

    if not (ROUND2 / "gradcam" / "gradcam_per_image.csv").exists():
        run_cmd([TORCH_PY, GPU_TASKS, "gradcam", "--script03b", SCRIPT_03B, "--manifest", ROUND2 / "manifests" / "gradcam_manifest.csv",
                 "--exp1b-dir", EXP1B_DIR, "--model", key, "--out", ROUND2 / "gradcam"], "G_gradcam.log")
    res = pd.read_csv(ROUND2 / "gradcam" / "gradcam_per_image.csv")
    g = gm.merge(res.drop(columns=["prob_malignant"], errors="ignore"), on="image_path")
    g = g[g.status == "ok"].copy()
    g["pred"] = (g.prob_malignant >= thr).astype(int)
    g["outcome"] = np.select([(g.label == 1) & (g.pred == 1), (g.label == 0) & (g.pred == 0), (g.label == 0) & (g.pred == 1)], ["TP", "TN", "FP"], "FN")
    summ = g.groupby("outcome").agg(images=("image_path", "size"), mean_cam_mass_outside_sector=("cam_mass_outside_sector", "mean"),
                                    median_cam_mass_outside_sector=("cam_mass_outside_sector", "median"),
                                    mean_area_outside_sector=("sector_area_outside", "mean"), share_peak_outside_sector=("peak_outside_sector", "mean")).reset_index()
    tot = dict(outcome="ALL", images=len(g), mean_cam_mass_outside_sector=g.cam_mass_outside_sector.mean(), median_cam_mass_outside_sector=g.cam_mass_outside_sector.median(),
               mean_area_outside_sector=g.sector_area_outside.mean(), share_peak_outside_sector=g.peak_outside_sector.mean())
    summ = pd.concat([summ, pd.DataFrame([tot])]).round(3)
    save_table(summ, "G1_gradcam_attribution", "Grad-CAM mass outside the detected ultrasound sector (EfficientNet-B3, val+test)",
               note="Expected mass under spatially uniform attribution equals the area outside the sector.")
    g.to_csv(ROUND2 / "tables" / "G2_gradcam_per_image.csv", index=False)
    # Figure: 8 test panels, 2 per outcome, readable captions
    cams = np.load(ROUND2 / "gradcam" / "cams.npz")["cams"]; idx_of = {p: i for i, p in enumerate(gm.image_path)}
    sel = pd.concat([g[(g.partition == "test") & (g.outcome == o)].sort_values("prob_malignant", ascending=(o in ("TN", "FN"))).head(2) for o in ["TP", "TN", "FP", "FN"]])
    fig, axes = plt.subplots(2, 4, figsize=(9, 5.2))
    for ax, (_, r) in zip(axes.ravel(), sel.iterrows()):
        base = np.asarray(Image.open(r.image_path).convert("L").resize((224, 224)), float)
        ax.imshow(base, cmap="gray"); ax.imshow(cams[idx_of[r.image_path]].astype(float), cmap="jet", alpha=0.35, vmin=0, vmax=1)
        ax.add_patch(Rectangle((r.box_x0 * 224, r.box_y0 * 224), (r.box_x1 - r.box_x0) * 224, (r.box_y1 - r.box_y0) * 224, fill=False, ec="white", lw=1, ls="--"))
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{r.outcome} · true {'PTC' if r.label else 'benign'}\np = {r.prob_malignant:.2f} · outside sector {r.cam_mass_outside_sector:.0%}", fontsize=7.5)
    for ax in axes.ravel()[len(sel):]: ax.axis("off")
    fig.tight_layout(); fig.savefig(ROUND2 / "figures" / "Fig_gradcam_ultrasound.png"); plt.close(fig)

## Stage I — Feature distributions across cohorts

In [ ]:
def mmd_rbf(X, Y, n_perm=500, max_n=300, seed=SEED):
    rng = np.random.default_rng(seed)
    X = X[rng.choice(len(X), min(max_n, len(X)), replace=False)]; Y = Y[rng.choice(len(Y), min(max_n, len(Y)), replace=False)]
    Z = np.vstack([X, Y]); n = len(X)
    D = ((Z[:, None, :] - Z[None, :, :]) ** 2).sum(-1); sigma2 = np.median(D[D > 0])
    K = np.exp(-D / sigma2)
    def stat(ix, iy):
        return K[np.ix_(ix, ix)].mean() + K[np.ix_(iy, iy)].mean() - 2 * K[np.ix_(ix, iy)].mean()
    obs = stat(np.arange(n), np.arange(n, len(Z)))
    perm = []
    for _ in range(n_perm):
        pz = rng.permutation(len(Z)); perm.append(stat(pz[:n], pz[n:]))
    return float(obs), float((np.sum(np.array(perm) >= obs) + 1) / (n_perm + 1))

with stage("I", "Feature distributions (t-SNE, MMD)") as go:
  if go:
    fm = pd.read_csv(FROZEN_MANIFEST, low_memory=False)
    parts_ = [fm[fm.modality == "ultrasound"][["image_path", "label", "split"]].assign(dataset=lambda d: "Pang " + d.split)]
    for name in ["TN3K_test", "DDTI_original"]:
        f = ROUND2 / "manifests" / f"external_{name}.csv"
        if f.exists(): parts_.append(pd.read_csv(f)[["image_path", "label"]].assign(dataset=name))
    em = pd.concat(parts_, ignore_index=True); em.to_csv(ROUND2 / "manifests" / "embedding_manifest.csv", index=False)
    if not (ROUND2 / "embeddings" / "embeddings.npy").exists():
        run_cmd([TORCH_PY, GPU_TASKS, "embed", "--script03b", SCRIPT_03B, "--manifest", ROUND2 / "manifests" / "embedding_manifest.csv",
                 "--exp1b-dir", EXP1B_DIR, "--model", "efficientnet_b3", "--out", ROUND2 / "embeddings", "--batch-size", BATCH_SIZE], "I_embed.log")
    F = np.load(ROUND2 / "embeddings" / "embeddings.npy"); idx = pd.read_csv(ROUND2 / "embeddings" / "embeddings_index.csv")
    Z = PCA(n_components=min(50, F.shape[1], len(F) - 1), random_state=SEED).fit_transform(StandardScaler().fit_transform(F))
    idx["group"] = np.where(idx.dataset.str.startswith("Pang"), "Pang (internal)", idx.dataset)
    groups = {gname: Z[idx.group.values == gname] for gname in idx.group.unique()}
    rows = []
    if "Pang (internal)" in groups:
        tr = Z[(idx.dataset == "Pang train").values]; te = Z[(idx.dataset == "Pang test").values]
        if len(tr) and len(te):
            s, pv = mmd_rbf(tr, te); rows.append(dict(comparison="Pang train vs Pang test (reference)", mmd2=round(s, 4), permutation_p=round(pv, 4)))
    for a, b in itertools.combinations(sorted(groups), 2):
        s, pv = mmd_rbf(groups[a], groups[b]); rows.append(dict(comparison=f"{a} vs {b}", mmd2=round(s, 4), permutation_p=round(pv, 4)))
    stats = []
    for gname, d in idx.groupby("group"):
        sz = [Image.open(p).size for p in d.image_path]; gray = [np.asarray(Image.open(p).convert("L").resize((128, 128)), float) for p in d.image_path]
        stats.append(dict(cohort=gname, images=len(d), median_width=int(np.median([s[0] for s in sz])), median_height=int(np.median([s[1] for s in sz])),
                          mean_intensity=round(float(np.mean([g.mean() for g in gray])), 1), mean_contrast_sd=round(float(np.mean([g.std() for g in gray])), 1),
                          share_near_black=round(float(np.mean([(g < 10).mean() for g in gray])), 3)))
    save_table(pd.DataFrame(rows), "I1_mmd", "Maximum mean discrepancy between cohorts (EfficientNet-B3 features, PCA-50, RBF kernel, 500 permutations)")
    save_table(pd.DataFrame(stats), "I2_image_statistics", "Acquisition-level image statistics by cohort")
    emb2 = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca").fit_transform(Z)
    fig, ax = plt.subplots(figsize=(5.2, 4.4)); colors = {"Pang (internal)": "#1f5fa8", "TN3K_test": "#d9822b", "DDTI_original": "#2e9e6b"}
    for gname in groups:
        mk = idx.group.values == gname
        ax.scatter(emb2[mk, 0], emb2[mk, 1], s=7, alpha=0.6, c=colors.get(gname, "#777"), label=f"{gname} (n={mk.sum()})", linewidths=0)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel("t-SNE 1"); ax.set_ylabel("t-SNE 2"); ax.legend(frameon=False, loc="best", markerscale=2)
    fig.tight_layout(); fig.savefig(ROUND2 / "figures" / "Fig_tsne_cohorts.png"); plt.close(fig)
    pd.DataFrame({"dataset": idx.dataset, "tsne1": emb2[:, 0], "tsne2": emb2[:, 1]}).to_csv(ROUND2 / "tables" / "I3_tsne_source.csv", index=False)

## Stage B — Grouped 5-fold CV for all backbones (your script 03b, unchanged)
Resumable per modality and backbone. ConvNeXt-Small already exists from August and is not re-run.

In [ ]:
FOLD_DECISION = ROUND2 / "logs" / "B_fold_decision.json"

def cv_results_dir():
    """Directory holding the grouped-CV runs used by stages C-E and K (decided once in stage B)."""
    if FOLD_DECISION.exists():
        return Path(json.loads(FOLD_DECISION.read_text())["cv_dir"])
    return CV_DIR

with stage("B", "Grouped 5-fold CV, all backbones") as go:
  if go:
    if sha256_file(FROZEN_MANIFEST) != FROZEN_SHA256:
        raise RuntimeError("Frozen manifest SHA mismatch — refusing to train")
    bk = ROUND2 / "cv_backup"; bk.mkdir(exist_ok=True)
    for f in ["environment.json", "master_cv_summary.csv", "fold_assignment.csv"]:
        if (CV_DIR / f).exists() and not (bk / f).exists():
            shutil.copy2(CV_DIR / f, bk / f)
    if not FOLD_DECISION.exists():
        # Fold membership of StratifiedGroupKFold depends on the scikit-learn version. Reuse the August ConvNeXt-Small
        # run only if 03b in THIS environment reproduces the saved folds exactly; otherwise refit all backbones together.
        r = subprocess.run([str(TORCH_PY), str(GPU_TASKS), "check-folds", "--script03b", str(SCRIPT_03B), "--manifest", str(FROZEN_MANIFEST),
                            "--fold-assignment", str(bk / "fold_assignment.csv")], capture_output=True, text=True)
        if r.returncode not in (0, 3):
            raise RuntimeError("fold check could not run (this is not a fold mismatch) — fix and re-run stage B:\n" + (r.stdout + r.stderr)[-2000:])
        match = r.returncode == 0
        FOLD_DECISION.write_text(json.dumps({"folds_reproduced": match, "cv_dir": str(CV_DIR if match else ROUND2 / "grouped_cv_refit"),
                                             "check_output": (r.stdout + r.stderr)[-1500:]}, indent=2))
    decision = json.loads(FOLD_DECISION.read_text()); OUT_CV = Path(decision["cv_dir"])
    note_md(f"- Fold check: {'saved August folds reproduced exactly; August ConvNeXt-Small run reused' if decision['folds_reproduced'] else 'saved August folds NOT reproduced in this environment; all five backbones re-run in ' + str(OUT_CV) + ' so every model shares identical folds'}.  \n  `{decision['check_output'].strip()[-300:]}`")
    reuse = decision["folds_reproduced"]
    for mo in ["ultrasound", "cytology"]:
        for key in MODELS:
            if (OUT_CV / mo / key / "cv_summary.json").exists():
                print(f"[done] {mo}/{key}"); continue
            try:
                run_cmd([TORCH_PY, SCRIPT_03B, "--models", key, "--modality", mo, "--out", OUT_CV,
                         "--batch-size", BATCH_SIZE, "--num-workers", NUM_WORKERS], f"B_cv_{mo}_{key}.log")
            finally:
                for f in ["environment.json", "master_cv_summary.csv", "fold_assignment.csv"]:
                    if (OUT_CV / f).exists():
                        shutil.copy2(OUT_CV / f, ROUND2 / "logs" / f"B_{Path(f).stem}_{mo}_{key}{Path(f).suffix}")
                    if reuse and (bk / f).exists():   # never let a partial call overwrite the August canonical files
                        shutil.copy2(bk / f, OUT_CV / f)
            fa_mo = ROUND2 / "manifests" / f"cv_fold_assignment_{mo}.csv"
            if not reuse and not fa_mo.exists():
                src = ROUND2 / "logs" / f"B_fold_assignment_{mo}_{key}.csv"
                if src.exists():
                    x = pd.read_csv(src); x[x.modality.astype(str).str.lower().eq(mo)].to_csv(fa_mo, index=False)
    if reuse and (bk / "fold_assignment.csv").exists():
        x = pd.read_csv(bk / "fold_assignment.csv")
        for mo in ["ultrasound", "cytology"]:
            x[x.modality.astype(str).str.lower().eq(mo)].to_csv(ROUND2 / "manifests" / f"cv_fold_assignment_{mo}.csv", index=False)
    summ = [json.loads((OUT_CV / mo / k / "cv_summary.json").read_text()) for mo in ["ultrasound", "cytology"] for k in MODELS
            if (OUT_CV / mo / k / "cv_summary.json").exists()]
    save_table(pd.DataFrame(summ).drop(columns=["manifest_sha256"], errors="ignore").round(4), "B1_cv_summary_from_03b", "Grouped 5-fold CV summaries written by 03b")

## Stage C — CV ensemble (thresholds from inner validation only) and fold-level tables

In [ ]:
def fold_leak_share(o):
    """Share of holdout images whose case (source_group) also has images in that fold's training folds."""
    shares = []
    for f in sorted(o.outer_fold.unique()):
        hold = o[o.outer_fold == f]; train_groups = set(o[o.outer_fold != f].source_group)
        shares.append(hold.source_group.isin(train_groups).mean())
    return float(np.mean(shares))

def cv_tables(o_by_model, thr_by_model_fold, mo, tag=""):
    fold_rows, summ_rows = [], []
    for k, o in o_by_model.items():
        conf = np.zeros(4, int)
        for f in sorted(o.outer_fold.unique()):
            h = o[o.outer_fold == f]; thr = thr_by_model_fold[(k, f)]
            m = full_metrics(h.label, h.prob_malignant, thr); conf += [m["tn"], m["fp"], m["fn"], m["tp"]]
            fold_rows.append(dict(modality=mo, model=DISPLAY.get(k, k) + tag, fold=int(f), n=m["n"], cases=int(h.source_group.nunique()),
                                  auroc=round(m["auroc"], 4), auprc=round(m["auprc"], 4), threshold_inner_val=round(thr, 4),
                                  sensitivity=round(m["sensitivity"], 3), specificity=round(m["specificity"], 3), brier=round(m["brier"], 4)))
        fr = pd.DataFrame([r for r in fold_rows if r["model"] == DISPLAY.get(k, k) + tag])
        pm = full_metrics(o.label, o.prob_malignant); ci = boot_ci(o.label, o.prob_malignant, o.component_id)
        tn, fp, fn, tp = conf
        summ_rows.append(dict(modality=mo, model=DISPLAY.get(k, k) + tag, dev_images=len(o), dev_cases=int(o.source_group.nunique()),
                              fold_auroc_mean_sd=f"{fr.auroc.mean():.3f} ± {fr.auroc.std(ddof=1):.3f}",
                              fold_auroc_range=f"{fr.auroc.min():.3f}–{fr.auroc.max():.3f}",
                              fold_auprc_mean_sd=f"{fr.auprc.mean():.3f} ± {fr.auprc.std(ddof=1):.3f}",
                              pooled_oof_auroc=ci_str(pm["auroc"], ci["auroc"]), pooled_oof_auprc=ci_str(pm["auprc"], ci["auprc"]),
                              pooled_brier=round(pm["brier"], 3), pooled_ece15=round(pm["ece15"], 3),
                              sensitivity_at_fold_thresholds=round(tp / (tp + fn), 3) if tp + fn else np.nan,
                              specificity_at_fold_thresholds=round(tn / (tn + fp), 3) if tn + fp else np.nan,
                              within_cv_case_leakage=round(fold_leak_share(o), 3)))
    return fold_rows, summ_rows

with stage("C", "CV ensemble and fold-level tables") as go:
  if go:
    CVR = cv_results_dir(); print("grouped-CV results from:", CVR)
    all_fold, all_summ, repro = [], [], []
    for mo in ["ultrasound", "cytology"]:
        avail = [k for k in MODELS if (CVR / mo / k / "cv_summary.json").exists()]
        fa = ROUND2 / "manifests" / f"cv_fold_assignment_{mo}.csv"
        if not fa.exists():
            raise RuntimeError(f"{fa} missing — run stage B first")
        if not avail: print(f"{mo}: no CV results yet"); continue
        need = [k for k in avail if not all((ROUND2 / "cv_innerval" / mo / k / f"fold_{f}_innerval_predictions.csv").exists() for f in range(1, 6))]
        if need:
            run_cmd([TORCH_PY, GPU_TASKS, "cv-innerval", "--script03b", SCRIPT_03B, "--manifest", FROZEN_MANIFEST, "--cv-dir", CVR,
                     "--fold-assignment", fa, "--modality", mo, "--models", *need, "--out", ROUND2 / "cv_innerval", "--batch-size", BATCH_SIZE], f"C_innerval_{mo}.log")
        oofs, thr, inner = {}, {}, {}
        for k in avail:
            o = pd.read_csv(CVR / mo / k / "oof_predictions.csv", low_memory=False); oofs[k] = o
            fmx = pd.read_csv(CVR / mo / k / "fold_metrics.csv")
            for f in range(1, 6):
                t03 = float(fmx.loc[fmx.outer_fold == f, "threshold_from_inner_val"].iloc[0]); thr[(k, f)] = t03
                iv = pd.read_csv(ROUND2 / "cv_innerval" / mo / k / f"fold_{f}_innerval_predictions.csv"); inner[(k, f)] = iv
                repro.append(dict(modality=mo, model=DISPLAY[k], fold=f, threshold_03b=t03, threshold_recomputed=float(iv.recomputed_threshold.iloc[0]),
                                  abs_diff=abs(t03 - float(iv.recomputed_threshold.iloc[0]))))
        fr, sr = cv_tables(oofs, thr, mo); all_fold += fr; all_summ += sr
        if len(avail) == len(MODELS):
            base = oofs[MODELS[0]][["image_path", "label", "source_group", "component_id", "outer_fold"]].copy()
            for k in MODELS:
                o = oofs[k][["image_path", "label", "outer_fold", "prob_malignant"]].rename(columns={"prob_malignant": f"prob_{k}", "label": f"label_{k}", "outer_fold": f"fold_{k}"})
                base = base.merge(o, on="image_path", how="inner", validate="one_to_one")
                assert (base[f"label_{k}"] == base.label).all() and (base[f"fold_{k}"] == base.outer_fold).all(), f"{mo}/{k}: OOF files disagree"
            assert len(base) == len(oofs[MODELS[0]]), f"{mo}: OOF image sets differ across backbones"
            base["prob_malignant"] = base[[f"prob_{k}" for k in MODELS]].mean(axis=1)
            for f in range(1, 6):
                iv = None
                for k in MODELS:
                    x = inner[(k, f)][["image_path", "label", "prob_malignant"]].rename(columns={"prob_malignant": f"p_{k}"})
                    iv = x if iv is None else iv.merge(x.drop(columns="label"), on="image_path", validate="one_to_one")
                thr[("ensemble", f)] = youden(iv.label, iv[[f"p_{k}" for k in MODELS]].mean(axis=1))
            base.to_csv(ROUND2 / "predictions" / f"cv_{mo}_ensemble_oof.csv", index=False)
            fr, sr = cv_tables({"ensemble": base}, thr, mo); all_fold += fr; all_summ += sr
    save_table(pd.DataFrame(all_summ), "C1_cv_summary", "Grouped 5-fold CV on the development set (train+val); frozen test untouched",
               note="Groups = duplicate-connected case components. Fold thresholds are Youden points on each fold's inner validation split; "
                    "pooled OOF CIs resample components. within_cv_case_leakage = share of holdout images whose case appears in training folds.")
    save_table(pd.DataFrame(all_fold), "C2_cv_fold_level", "Fold-level results (all backbones and ensemble)", show=False)
    if repro:
        r = pd.DataFrame(repro); save_table(r.groupby(["modality", "model"]).abs_diff.max().reset_index(name="max_abs_threshold_difference"),
                                            "C3_threshold_reproducibility", "Inner-validation thresholds recomputed from checkpoints vs values logged by 03b")

## Stage D — Leakage demonstration: identical code, image-level vs case-grouped folds (cytology)

In [ ]:
with stage("D", "Leakage demonstration") as go:
  if go:
    out = ROUND2 / "imagelevel_cv"; LEAK_MODELS = ["convnext_small", "efficientnet_b3"]
    for k in LEAK_MODELS:
        if not (out / "cytology" / k / "cv_summary.json").exists():
            run_cmd([TORCH_PY, PATCHED_IMAGELEVEL, "--models", k, "--modality", "cytology", "--out", out,
                     "--batch-size", BATCH_SIZE, "--num-workers", NUM_WORKERS], f"D_imagelevel_{k}.log")
    rows, pairs = [], []
    for k in LEAK_MODELS:
        per_design = {}
        for design, base in [("case-grouped folds", cv_results_dir()), ("image-level folds", out)]:
            d = base / "cytology" / k
            if not (d / "cv_summary.json").exists(): continue
            o = pd.read_csv(d / "oof_predictions.csv", low_memory=False); fmx = pd.read_csv(d / "fold_metrics.csv")
            pm = full_metrics(o.label, o.prob_malignant); ci = boot_ci(o.label, o.prob_malignant, o.component_id)
            rows.append(dict(model=DISPLAY[k], design=design, fold_auroc_mean_sd=f"{fmx.auroc.mean():.3f} ± {fmx.auroc.std(ddof=1):.3f}",
                             pooled_oof_auroc=ci_str(pm["auroc"], ci["auroc"]), pooled_brier=round(pm["brier"], 3),
                             within_cv_case_leakage=round(fold_leak_share(o), 3)))
            per_design[design] = fmx.set_index("outer_fold").auroc
        if len(per_design) == 2:
            for f in per_design["case-grouped folds"].index:
                pairs.append(dict(model=DISPLAY[k], fold=int(f), case_grouped=per_design["case-grouped folds"][f], image_level=per_design["image-level folds"][f]))
    save_table(pd.DataFrame(rows), "D1_leakage_demonstration", "Same code, data and seeds; only the fold grouping unit differs (cytology)",
               note="Image-level folds let blocks from the same case fall on both sides of every fold, reproducing the original design.")
    if pairs: save_table(pd.DataFrame(pairs).round(4), "D2_leakage_fold_pairs", "Fold AUROC by grouping unit", show=False)

## Stage E — Shortcut control (ultrasound): what can the model learn from outside the sector?

In [ ]:
with stage("E", "Shortcut control") as go:
  if go:
    key = "efficientnet_b3"; outs = {m: ROUND2 / "shortcut_cv" / m for m in ["surround_only", "center_only"]}
    for mode, out in outs.items():
        if not (out / "ultrasound" / key / "cv_summary.json").exists():
            run_cmd([TORCH_PY, PATCHED_SHORTCUT, "--models", key, "--modality", "ultrasound", "--out", out,
                     "--batch-size", BATCH_SIZE, "--num-workers", NUM_WORKERS], f"E_shortcut_{mode}.log", env_extra={"THYRO_MASK_MODE": mode})
    rows, folds = [], []
    for cond, base in [("full image", cv_results_dir()), ("surround only (sector blacked out)", outs["surround_only"]), ("sector only (surround blacked out)", outs["center_only"])]:
        d = base / "ultrasound" / key
        if not (d / "cv_summary.json").exists(): continue
        o = pd.read_csv(d / "oof_predictions.csv", low_memory=False); fmx = pd.read_csv(d / "fold_metrics.csv")
        pm = full_metrics(o.label, o.prob_malignant); ci = boot_ci(o.label, o.prob_malignant, o.component_id)
        rows.append(dict(input=cond, fold_auroc_mean_sd=f"{fmx.auroc.mean():.3f} ± {fmx.auroc.std(ddof=1):.3f}",
                         fold_aurocs=", ".join(f"{v:.3f}" for v in fmx.sort_values("outer_fold").auroc),
                         pooled_oof_auroc=ci_str(pm["auroc"], ci["auroc"]), ci_excludes_chance=bool(ci["auroc"][0] > 0.5)))
        for _, r in fmx.iterrows(): folds.append(dict(input=cond, fold=int(r.outer_fold), auroc=r.auroc))
    save_table(pd.DataFrame(rows), "E1_shortcut_control", "EfficientNet-B3 grouped 5-fold CV with the sector or the surround removed (same folds and seeds)",
               note="If the surround-only model discriminates (CI above 0.5), label information is recoverable from outside the image of the thyroid.")
    pd.DataFrame(folds).to_csv(ROUND2 / "tables" / "E2_shortcut_folds.csv", index=False)

## Stage J — Publication figures

In [ ]:
def _box(ax, x, y, w, h, text, fc="#eef3f9", ec="#1f4e79", fs=7.2, bold_first=True):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.008,rounding_size=0.012", fc=fc, ec=ec, lw=1))
    lines = text.split("\n")
    ax.text(x + w / 2, y + h - 0.018, lines[0], ha="center", va="top", fontsize=fs + 0.6, weight="bold" if bold_first else "normal", color="#10263d")
    if len(lines) > 1: ax.text(x + w / 2, y + h - 0.058, "\n".join(lines[1:]), ha="center", va="top", fontsize=fs, color="#10263d", linespacing=1.35)

def _arrow(ax, x0, y0, x1, y1):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", lw=1, color="#1f4e79"))

with stage("J", "Figures") as go:
  if go:
    made = []
    try:  # Figure 1 flow diagram
        flow = json.loads((ROUND2 / "tables" / "A_flow.json").read_text())
        fz = pd.DataFrame(flow["frozen"]); get = lambda mo, sp, c: int(fz[(fz.modality == mo) & (fz.split == sp)][c].iloc[0])
        ext_lines = []
        for name, label in [("TN3K_test", "TN3K official test"), ("DDTI_original", "DDTI original")]:
            f = ROUND2 / "manifests" / f"external_{name}.csv"
            if f.exists():
                e = pd.read_csv(f); ext_lines.append(f"{label}: {len(e)} images" + (f", {e.case_id.nunique()} cases" if "DDTI" in name else ""))
        fig, ax = plt.subplots(figsize=(7.2, 4.8)); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
        _box(ax, 0.14, 0.80, 0.50, 0.17, f"Pang et al. archive (as released)\n{flow['cyto_blocks']} cytology blocks ({flow['cyto_cases']} cases)\n{flow['us_images']} ultrasound images ({flow['us_cases']} cases)")
        _box(ax, 0.70, 0.815, 0.28, 0.14, f"Excluded\n{flow['excluded_us']} ultrasound images in\ncross-label duplicate groups", fc="#fbf1e6", ec="#b5651d")
        _box(ax, 0.02, 0.46, 0.45, 0.17, f"Cytology (2–7 blocks per case)\nimages: train {get('cytology','train','images')} · val {get('cytology','val','images')} · test {get('cytology','test','images')}\ncases: train {get('cytology','train','cases')} · val {get('cytology','val','cases')} · test {get('cytology','test','cases')}")
        _box(ax, 0.53, 0.46, 0.45, 0.17, f"Ultrasound (1 image per case)\ntrain {get('ultrasound','train','images')} · val {get('ultrasound','val','images')} · test {get('ultrasound','test','images')}\nbenign/PTC in test: {get('ultrasound','test','benign')}/{get('ultrasound','test','ptc')}")
        _box(ax, 0.02, 0.08, 0.45, 0.22, "Internal evaluation\ngrouped 5-fold CV on train+val (both modalities)\nfrozen case-disjoint test set\nthresholds selected on validation only")
        _box(ax, 0.53, 0.08, 0.45, 0.22, "External evaluation\nfrozen ultrasound models, no re-training\n" + ("\n".join(ext_lines) if ext_lines else "not run"))
        _arrow(ax, 0.64, 0.885, 0.70, 0.885)            # archive -> excluded
        _arrow(ax, 0.30, 0.80, 0.245, 0.63)            # archive -> cytology
        _arrow(ax, 0.48, 0.80, 0.755, 0.63)            # archive -> ultrasound
        _arrow(ax, 0.245, 0.46, 0.245, 0.30)           # cytology -> internal
        _arrow(ax, 0.60, 0.46, 0.44, 0.30)             # ultrasound -> internal
        _arrow(ax, 0.755, 0.46, 0.755, 0.30)           # ultrasound -> external
        fig.savefig(ROUND2 / "figures" / "Fig1_cohort_flow.png", bbox_inches="tight"); plt.close(fig); made.append("Fig1")
    except Exception as e: print("Fig1 skipped:", e)

    try:  # CV fold AUROC by backbone
        c2 = pd.read_csv(ROUND2 / "tables" / "C2_cv_fold_level.csv")
        fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), sharey=True)
        for ax, mo in zip(axes, ["cytology", "ultrasound"]):
            d = c2[c2.modality == mo]; order = [DISPLAY[k] for k in MODELS] + [DISPLAY["ensemble"]]; order = [o for o in order if o in set(d.model)]
            for i, mname in enumerate(order):
                v = d[d.model == mname].auroc.values
                ax.scatter(np.full(len(v), i) + np.linspace(-0.12, 0.12, len(v)), v, s=14, color="#1f5fa8" if mname != DISPLAY["ensemble"] else "#b5651d", zorder=3)
                ax.plot([i - 0.25, i + 0.25], [v.mean()] * 2, color="#10263d", lw=1.4)
            ax.axhline(0.5, color="#999", lw=0.8, ls="--"); ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=35, ha="right")
            ax.set_title(mo.capitalize()); ax.set_ylim(0.3, 1.02)
        axes[0].set_ylabel("Fold AUROC (grouped 5-fold CV)")
        fig.tight_layout(); fig.savefig(ROUND2 / "figures" / "Fig2_cv_fold_auroc.png"); plt.close(fig); made.append("Fig2")
    except Exception as e: print("Fig2 skipped:", e)

    try:  # leakage
        d2 = pd.read_csv(ROUND2 / "tables" / "D2_leakage_fold_pairs.csv")
        fig, ax = plt.subplots(figsize=(3.6, 3.0))
        for i, (mname, d) in enumerate(d2.groupby("model")):
            for _, r in d.iterrows():
                ax.plot([i - 0.15, i + 0.15], [r.case_grouped, r.image_level], color="#7a8ca3", lw=0.8)
            ax.scatter(np.full(len(d), i - 0.15), d.case_grouped, s=14, color="#1f5fa8", label="case-grouped folds" if i == 0 else None, zorder=3)
            ax.scatter(np.full(len(d), i + 0.15), d.image_level, s=14, color="#c0392b", label="image-level folds" if i == 0 else None, zorder=3)
        ax.set_xticks(range(d2.model.nunique())); ax.set_xticklabels(sorted(d2.model.unique())); ax.set_ylabel("Fold AUROC (cytology)")
        ax.legend(frameon=False, loc="lower right"); ax.set_ylim(0.6, 1.01)
        fig.tight_layout(); fig.savefig(ROUND2 / "figures" / "Fig3_leakage_demonstration.png"); plt.close(fig); made.append("Fig3")
    except Exception as e: print("Fig3 skipped:", e)

    try:  # frozen test ROC + reliability
        fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2))
        for mo, col in [("cytology", "#1f5fa8"), ("ultrasound", "#b5651d")]:
            t = pd.read_csv(EXP1B_DIR / mo / "ensemble" / "test_predictions.csv"); fpr, tpr, _ = roc_curve(t.label, t.prob_malignant)
            axes[0].plot(fpr, tpr, color=col, lw=1.5, label=f"{mo} ensemble (AUROC {roc_auc_score(t.label, t.prob_malignant):.3f})")
        axes[0].plot([0, 1], [0, 1], "--", color="#999", lw=0.8); axes[0].set_xlabel("1 − specificity"); axes[0].set_ylabel("Sensitivity")
        axes[0].set_title("Frozen test set (image level)"); axes[0].legend(frameon=False, loc="lower right")
        rel = pd.read_csv(ROUND2 / "tables" / "F4_reliability_source.csv")
        for (mo, sc), d in rel.groupby(["modality", "scaling"]):
            axes[1].plot(d.mean_pred, d.frac_pos, marker="o", ms=3, lw=1, ls="-" if sc == "after" else ":",
                         color="#1f5fa8" if mo == "cytology" else "#b5651d", label=f"{mo}, {'after' if sc == 'after' else 'before'} scaling")
        axes[1].plot([0, 1], [0, 1], "--", color="#999", lw=0.8); axes[1].set_xlabel("Mean predicted probability"); axes[1].set_ylabel("Observed PTC fraction")
        axes[1].set_title("Calibration (temperature fitted on validation)"); axes[1].legend(frameon=False, fontsize=7)
        fig.tight_layout(); fig.savefig(ROUND2 / "figures" / "Fig4_frozen_test_roc_calibration.png"); plt.close(fig); made.append("Fig4")
    except Exception as e: print("Fig4 skipped:", e)

    try:  # shortcut control
        e2 = pd.read_csv(ROUND2 / "tables" / "E2_shortcut_folds.csv")
        fig, ax = plt.subplots(figsize=(3.8, 3.0)); conds = list(dict.fromkeys(e2.input))
        for i, c in enumerate(conds):
            v = e2[e2.input == c].auroc.values
            ax.scatter(np.full(len(v), i) + np.linspace(-0.1, 0.1, len(v)), v, s=16, color="#1f5fa8", zorder=3); ax.plot([i - 0.25, i + 0.25], [v.mean()] * 2, color="#10263d", lw=1.4)
        ax.axhline(0.5, color="#999", lw=0.8, ls="--"); ax.set_xticks(range(len(conds)))
        ax.set_xticklabels([c.split(" (")[0] for c in conds]); ax.set_ylabel("Fold AUROC (ultrasound, EfficientNet-B3)"); ax.set_ylim(0.2, 1.0)
        fig.tight_layout(); fig.savefig(ROUND2 / "figures" / "Fig5_shortcut_control.png"); plt.close(fig); made.append("Fig5")
    except Exception as e: print("Fig5 skipped:", e)

    try:  # external ROC
        panels = [(n, c, t) for n, c, t in [("TN3K_test", "label", "TN3K official test (benign vs malignant)"),
                                            ("DDTI_original", "y_primary", "DDTI original (TI-RADS 2–3 vs 4–5)")]
                  if (ROUND2 / "predictions" / f"external_{n}.csv").exists()]
        if panels:
            fig, axes = plt.subplots(1, len(panels), figsize=(3.6 * len(panels), 3.2)); axes = np.atleast_1d(axes)
            for ax, (n, c, title) in zip(axes, panels):
                d = pd.read_csv(ROUND2 / "predictions" / f"external_{n}.csv"); d = d[d[c].isin([0, 1])]
                for k in MODELS + ["ensemble"]:
                    fpr, tpr, _ = roc_curve(d[c].astype(int), d[f"prob_{k}"])
                    ax.plot(fpr, tpr, lw=1.8 if k == "ensemble" else 0.9, color="#b5651d" if k == "ensemble" else None, alpha=1 if k == "ensemble" else 0.7,
                            label=f"{DISPLAY[k]} ({roc_auc_score(d[c].astype(int), d[f'prob_{k}']):.3f})")
                ax.plot([0, 1], [0, 1], "--", color="#999", lw=0.8); ax.set_title(title); ax.set_xlabel("1 − specificity"); ax.set_ylabel("Sensitivity")
                ax.legend(frameon=False, fontsize=6.5, loc="lower right")
            fig.tight_layout(); fig.savefig(ROUND2 / "figures" / "Fig6_external_roc.png"); plt.close(fig); made.append("Fig6")
    except Exception as e: print("Fig6 skipped:", e)
    note_md(f"- Figures written: {made} (+ Grad-CAM, t-SNE and supplementary sheets from stages A, G, I).")

## Stage K — Release package (GitHub/Zenodo) and the results zip to upload

In [ ]:
README_TEMPLATE = """# ThyroFuse revision — analysis release

Everything needed to reproduce the revised results. The development data (Pang et al., Mendeley) and external data (TN3K, DDTI) must be obtained from their distributors; only derived manifests are included.

| Folder | Contents | Reviewer items |
|---|---|---|
| manifests/ | frozen case-level manifest (+SHA-256), fold assignment, exclusions, duplicate groups, external manifests | R2-4, R3-1, R3-3, R3-5, R6-3, R6-7 |
| predictions/ | per-image probabilities: frozen test (all models), grouped-CV out-of-fold (all models + ensemble), image-level CV, shortcut control, external | R2-2, R2-4, R3-3, R5 |
| thresholds/ | every operating threshold and the data it was selected on | R2-5, R2-6 |
| tables/, figures/ | all reported tables with figure source data | R5, R6-10 |
| code/ | training (02, 03b), the two patched 03b variants, inference script, this notebook | R3-3, R5, R6-7 |
| environment/ | pip freeze and GPU details | R3-3 |
"""

with stage("K", "Release package and upload zip") as go:
  if go:
    rel = ROUND2 / "release"
    for sub in ["manifests", "predictions", "thresholds", "tables", "figures", "code", "environment"]:
        (rel / sub).mkdir(parents=True, exist_ok=True)
    def cp(src, dst_dir, name=None):
        src = Path(src)
        if src.exists(): shutil.copy2(src, rel / dst_dir / (name or src.name)); return True
        return False
    cp(FROZEN_MANIFEST, "manifests"); (rel / "manifests" / "04_frozen_manifest.sha256").write_text(sha256_file(FROZEN_MANIFEST))
    cp(EXCLUDED_CSV, "manifests")
    for mo in ["ultrasound", "cytology"]: cp(ROUND2 / "manifests" / f"cv_fold_assignment_{mo}.csv", "manifests")
    for f in (ROUND2 / "manifests").glob("external_*.csv"): cp(f, "manifests")
    cp(ROUND2 / "tables" / "A2_exact_duplicates.csv", "manifests", "duplicate_groups.csv")
    for mo in ["cytology", "ultrasound"]:
        for k in MODELS + ["ensemble"]:
            for part in ["val", "test"]:
                cp(EXP1B_DIR / mo / k / f"{part}_predictions.csv", "predictions", f"frozen_{mo}_{k}_{part}.csv")
            if k != "ensemble": cp(cv_results_dir() / mo / k / "oof_predictions.csv", "predictions", f"cv_{mo}_{k}_oof.csv")
    for f in (ROUND2 / "predictions").glob("*.csv"): cp(f, "predictions")
    for base, tag in [(ROUND2 / "imagelevel_cv", "imagelevel"), (ROUND2 / "shortcut_cv", "shortcut")]:
        for f in base.rglob("oof_predictions.csv"): cp(f, "predictions", f"{tag}_{'_'.join(f.relative_to(base).parts[:-1])}_oof.csv")
    thr_rows = []
    for mo in ["cytology", "ultrasound"]:
        for k in MODELS + ["ensemble"]:
            mj = EXP1B_DIR / mo / k / "metrics.json"
            if mj.exists():
                j = json.loads(mj.read_text()); thr_rows.append(dict(analysis="frozen split", modality=mo, model=k, selected_on="validation (Youden)",
                                                                     image_level=j.get("image_level_threshold_from_val"), case_level=j.get("source_group_threshold_from_val")))
    pd.DataFrame(thr_rows).to_csv(rel / "thresholds" / "frozen_thresholds.csv", index=False)
    if (ROUND2 / "tables" / "C2_cv_fold_level.csv").exists():
        pd.read_csv(ROUND2 / "tables" / "C2_cv_fold_level.csv").to_csv(rel / "thresholds" / "cv_fold_thresholds_inner_validation.csv", index=False)
    for f in (ROUND2 / "tables").glob("*"): cp(f, "tables")
    for f in (ROUND2 / "figures").glob("*.png"): cp(f, "figures")
    for f in [SCRIPT_02, SCRIPT_03B, PATCHED_IMAGELEVEL, PATCHED_SHORTCUT, GPU_TASKS]: cp(f, "code")
    nbs = [p for p in list(ROOT.glob("*.ipynb")) + list(Path.home().glob("Downloads/*.ipynb")) if "round2" in p.name.lower()]
    for p in nbs: cp(p, "code")
    try:
        (rel / "environment" / "pip_freeze.txt").write_text(subprocess.run([str(TORCH_PY), "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
        (rel / "environment" / "nvidia_smi.txt").write_text(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    except Exception as e:
        print("environment capture:", e)
    (rel / "README.md").write_text(README_TEMPLATE, encoding="utf-8")
    note_md(f"- Release folder ready: {rel} — push it to GitHub, tag a release, and archive it on Zenodo for a DOI.")

    STATUS["K"] = "done"; (ROUND2 / "logs" / "STATUS.json").write_text(json.dumps(STATUS, indent=2))
    up = ROUND2 / "REVISION_ROUND2_UPLOAD.zip"; cap = 28e6
    with zipfile.ZipFile(up, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
        def add(path, arc):
            if Path(path).exists() and sum(i.compress_size for i in z.infolist()) < cap: z.write(path, arc)
        add(RESULTS_MD, "RESULTS.md"); add(ROUND2 / "logs" / "STATUS.json", "STATUS.json")
        for f in sorted((ROUND2 / "tables").glob("*")): add(f, f"tables/{f.name}")
        for f in sorted((ROUND2 / "logs").glob("*.log")):
            lines = f.read_text(encoding="utf-8", errors="replace").splitlines()
            z.writestr(f"logs/{f.name}", "\n".join(lines[:40] + ["... [truncated] ..."] + lines[-250:] if len(lines) > 300 else lines))
        for f in sorted((ROUND2 / "manifests").glob("external_*.csv")): add(f, f"manifests/{f.name}")
        for f in sorted((ROUND2 / "predictions").glob("*.csv")): add(f, f"predictions/{f.name}")
        for f in sorted((ROUND2 / "figures").glob("*.png")): add(f, f"figures/{f.name}")
        for mo in ["cytology", "ultrasound"]:
            for k in MODELS:
                add(cv_results_dir() / mo / k / "cv_summary.json", f"cv/{mo}_{k}_cv_summary.json"); add(cv_results_dir() / mo / k / "fold_metrics.csv", f"cv/{mo}_{k}_fold_metrics.csv")
    print(f"\n>>> UPLOAD THIS FILE: {up}  ({up.stat().st_size / 1e6:.1f} MB)")
    print("stage status:", json.dumps(STATUS, indent=1))